# Sesión 7 — MBD

## Clasificación supervisada (II)

En esta sesión trabajaremos con un conjunto de productos para aprender a predecir su clase a partir de sus características. Compararemos tres enfoques: **árboles de inferencia condicional**, **random forest** y **máquinas de vectores soporte (SVM)**.

### ¿Qué aprenderás?

- Interpretar las reglas de un árbol y explorar cómo la **poda** cambia su complejidad.
- Evaluar un random forest con el error **fuera de bolsa (OOB)** y examinar la importancia de sus variables.
- Comparar kernels y ajustar hiperparámetros de SVM mediante **validación cruzada**.
- Leer matrices de confusión y comparar el acierto global con el acierto por clase, distinguiendo entrenamiento, validación y test.

A lo largo del notebook encontrarás ejemplos ejecutables, controles interactivos y **microprácticas** para completar huecos. Los ejercicios 7-1, 7-2 y 7-3 quedan para resolver al final de sus respectivos bloques.

### Cómo trabajar en VS Code

1. Selecciona el **kernel de Python del proyecto** y ejecuta las celdas en orden con **Shift + Enter**.
2. Usa **Variables** para inspeccionar datos y modelos, y **Esquema / Outline** para navegar por las secciones.
3. Ejecuta las celdas de los controles antes de utilizarlos; los controles anteriores permanecen disponibles mientras el kernel conserve sus variables.
4. Para repetir la sesión desde cero, usa **Reiniciar** y después **Ejecutar todo**. [Guía de notebooks en VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks).

## 7.0. Preparación

Selecciona el kernel del proyecto y ejecuta las primeras celdas en orden. La primera localiza la carpeta del proyecto y los scripts de esta sesión; después importamos las bibliotecas y fijamos `SEED = 12345` para repetir las particiones aleatorias.

Las funciones auxiliares se organizan en scripts independientes: `s7_evaluacion.py`, `s7_arboles.py`, `s7_poda.py`, `s7_bosques.py`, `s7_svm.py` y `s7_graficos.py`. El notebook conserva la carga de datos, los ajustes de cada ejemplo, la comparación de resultados y los ejercicios. Cada script contiene funciones reutilizables que reciben los datos y modelos como argumentos.

Las matrices de confusión muestran las predicciones en filas y las clases reales en columnas. Reservamos las particiones de test para las comparaciones de cada bloque; cuando una muestra se usa para escoger parámetros, la interpretación es exploratoria y no equivale a una evaluación final independiente.

In [ ]:
from pathlib import Path  # Permite construir rutas de archivos sin depender de macOS, Windows o Linux.
import sys  # Da acceso a sys.path, la lista de carpetas donde Python busca módulos.

for carpeta in (Path.cwd(), *Path.cwd().parents):  # Prueba la carpeta actual y, después, cada carpeta superior.
    if (carpeta / "datos" / "products.csv").is_file():  # Comprueba si aquí existe el archivo de datos esperado.
        PROJECT_PATH = carpeta  # Guarda como raíz la carpeta que contiene datos/products.csv.
        DATA_PATH = carpeta / "datos"  # Forma la ruta completa a la carpeta de datos.
        break  # Detiene la búsqueda al encontrar la raíz para no seguir subiendo.
else:  # Este bloque se ejecuta solo si el bucle terminó sin encontrar el archivo.
    raise FileNotFoundError("Abre el notebook dentro de la carpeta LaSalle_Estadistica.")  # Explica cómo corregir la ruta.

SESSION_PATH = PROJECT_PATH / "sesiones" / "sesion7"  # Localiza la carpeta con los scripts auxiliares de S7.
if str(SESSION_PATH) not in sys.path:  # Evita añadir la misma carpeta más de una vez.
    sys.path.insert(0, str(SESSION_PATH))  # Permite importar los scripts de S7 desde el notebook.

In [ ]:
# Esta instalación es opcional: descomenta la siguiente línea solo si el kernel no tiene estas bibliotecas.
# %pip install numpy pandas matplotlib scipy scikit-learn ipywidgets threadpoolctl ipykernel

In [ ]:
import numpy as np  # Importa NumPy con el alias np para operar con arrays y números.
import pandas as pd  # Importa pandas con el alias pd para leer y organizar tablas.
import matplotlib.pyplot as plt  # Importa las funciones de gráficos de Matplotlib.
import ipywidgets as widgets  # Importa controles interactivos, como deslizadores y menús.
from IPython.display import display  # Permite mostrar tablas, gráficos y controles en la salida.
from sklearn.ensemble import RandomForestClassifier  # Proporciona el algoritmo de random forest.
from sklearn.model_selection import KFold, GridSearchCV  # Divide datos en pliegues y busca hiperparámetros.
from threadpoolctl import threadpool_limits  # Limita hilos numéricos para mantener los cálculos estables.
from s7_evaluacion import confusion, evaluar_clasificador  # Importa la matriz de confusión y el resumen de métricas.
from s7_graficos import barras_clases  # Importa el gráfico que compara el acierto de las clases.
from s7_poda import arbol_entropia, trayectoria_poda, podar_a_hojas, validar_poda  # Importa ajuste, secuencia de poda y validación cruzada.
from s7_bosques import diagnostico_oob  # Importa el cálculo de votos, errores y cobertura OOB.
from s7_bosques import importancia_oob  # Importa la función que calcula importancia por permutación OOB y Gini.

# Pide a Jupyter que muestre los gráficos debajo de cada celda.
%matplotlib inline
# Configura la celda para mostrar todos los resultados, no solo el último.
%config InteractiveShell.ast_node_interactivity = "all"
plt.rcParams.update({"figure.figsize": (8, 4), "figure.dpi": 100})  # Ajusta el tamaño y resolución iniciales de las figuras.
pd.set_option("display.max_columns", 30)  # Permite ver hasta 30 columnas cuando se muestra una tabla.
SEED = 12345  # Fija una semilla para repetir los resultados aleatorios.

### Datos y variables

Cada fila representa un producto y cada columna contiene una característica o una etiqueta:

| Variable | Contenido |
| --- | --- |
| `id` | Identificador del producto; no describe sus características y se retirará antes de modelar. |
| `feat_1` … `feat_93` | 93 características numéricas que podrán utilizarse como predictores. |
| `target` | Clase conocida del producto que queremos aprender a predecir. |

Leemos el archivo separado por punto y coma. En la primera inspección comprobamos el tamaño, los nombres de las columnas y algunas filas; todavía no ajustamos ningún modelo.

In [ ]:
datos = pd.read_csv(DATA_PATH / "products.csv", sep=";", decimal=",")  # Lee el CSV; reconoce ; como separador y , como decimal.
datos.shape  # Muestra una tupla (filas, columnas) para conocer el tamaño de la tabla.
datos.head()  # Enseña las primeras cinco filas para revisar que la lectura parece correcta.

In [ ]:
datos.describe(include="all")  # Resume cada columna; include="all" también incluye variables no numéricas.
datos["target"].value_counts().sort_index()  # Cuenta productos por clase y ordena las etiquetas.

### Partición de entrenamiento y test

Separamos las filas en dos grupos: **entrenamiento** para que el algoritmo aprenda y **test** para comprobar sus predicciones en filas que no usó al ajustarse. Cada producto entra en entrenamiento con probabilidad 0,7; el resto queda en test.

La semilla hace que la asignación aleatoria se pueda repetir. Los árboles y bosques comparten esta partición para que sus resultados sean comparables. El bloque de SVM crea otra submuestra y otra partición, como se explica más adelante.

In [ ]:
d = datos.drop(columns="id")  # Quita el identificador para que no se use como característica predictiva.
predictores = list(d.columns.drop("target"))  # Guarda los nombres de las 93 características de entrada.
clases = np.sort(d["target"].unique())  # Extrae las etiquetas distintas y las ordena.
rng_arbol = np.random.default_rng(SEED)  # Crea un generador aleatorio reproducible con la semilla elegida.
train_sel = rng_arbol.choice([False, True], size=len(d), p=[0.3, 0.7])  # Marca cada fila para test (0,3) o train (0,7).
train = d.loc[train_sel].copy()  # Copia las filas marcadas como entrenamiento.
test = d.loc[~train_sel].copy()  # Copia las filas restantes; ~ invierte la máscara booleana.
X_train, y_train = train[predictores], train["target"]  # Separa predictores y etiquetas del entrenamiento.
X_test, y_test = test[predictores], test["target"]  # Separa predictores y etiquetas del test.
pd.Series({"Entrenamiento": len(train), "Test": len(test), "Predictores": len(predictores)})  # Resume tamaños de las particiones y número de entradas.

## 7.1. Árboles de inferencia condicional

Un árbol divide los productos en grupos cada vez más homogéneos. En cada nodo, la implementación de apoyo contrasta si cada predictor está asociado con la clase; corrige los contrastes porque examina muchas variables y escoge una división binaria cuando encuentra evidencia suficiente.

La implementación de `s7_arboles.py` está preparada para **predictores numéricos y una respuesta nominal**, como los de este archivo. El módulo contiene los cálculos estadísticos y la estructura del árbol; aquí veremos cómo se ajusta, se interpreta y se evalúa.

**Lectura de los resultados:** cada nodo muestra una regla, como `feat_12 ≤ corte`, y las hojas muestran la clase mayoritaria. La profundidad cuenta las divisiones desde la raíz. Limitarla produce árboles más sencillos, aunque puede reducir el detalle que aprenden.

[Fundamento de los árboles de inferencia condicional](https://www.zeileis.org/papers/Hothorn%2BHornik%2BZeileis-2006.pdf).

In [ ]:
from s7_arboles import ArbolCondicional  # Importa la clase que implementa el ajuste y la predicción del árbol.
from s7_graficos import dibujar_arbol, inspeccionar_nodo  # Importa las funciones para dibujar e inspeccionar árboles.

### Construir un árbol poco profundo

Primero ajustamos un árbol con `max_depth=3` utilizando únicamente los productos de entrenamiento. La **raíz tiene profundidad 0**: un producto puede atravesar como máximo tres divisiones antes de llegar a una hoja. El límite se aplica a cada camino, no al número total de divisiones del árbol. Algunas ramas pueden terminar antes si se cumple una condición de parada.

En cada nodo, el árbol considera solo los productos que han llegado hasta allí. Para cada predictor numérico, contrasta si la media es igual en todas las clases presentes. Si $\mu_{cj}$ es la media del predictor $j$ en la clase $c$, las hipótesis son:

$$
H_0: \mu_{1j}=\mu_{2j}=\cdots=\mu_{Cj}
\qquad\text{frente a}\qquad
H_1: \text{no todas las medias de clase son iguales}.
$$

El estadístico compara la variación **entre las medias de las clases** con la variación **total** del predictor. En esta implementación se calcula y se compara con una distribución chi-cuadrado aproximada:

$$
T_j=\frac{(n-1)\,SS_{\text{entre},j}}{SS_{\text{total},j}},
\qquad
T_j\overset{H_0}{\approx}\chi^2_{C-1}.
$$

A partir de ahí se obtiene el p-valor: la probabilidad, suponiendo que $H_0$ sea cierta, de observar un estadístico al menos tan extremo como $T_j$:

$$
p_j=P\!\left(\chi^2_{C-1}\ge T_j\mid H_0\right).
$$

Como se examinan $m$ predictores, el árbol ajusta los p-valores por ese número de comparaciones. La corrección aplicada y la regla de decisión se pueden resumir así:

$$
p_{j,\text{aj}}=1-(1-p_j)^m,
\qquad
\text{se considera dividir si }\min_j p_{j,\text{aj}}\le\alpha=0.05.
$$

El p-valor **no** es la probabilidad de que $H_0$ sea verdadera: indica cuán incompatibles serían los datos observados con esa hipótesis. Si ningún predictor supera el criterio de significación, el nodo queda como hoja.

Una vez seleccionado un predictor, el árbol busca un valor de corte que separe las clases respetando los tamaños mínimos de las ramas. En este ejemplo, `splitstat="maximum"` sirve para **puntuar y elegir el corte**, no para hacer otro contraste. Para entender su puntuación, ordenamos los productos por el predictor y consideramos un corte después de las primeras $k$ observaciones. Si $L_c(k)$ es el recuento observado de la clase $c$ a la izquierda y $\pi_c$ su proporción en el nodo, la diferencia respecto al recuento esperado bajo una mezcla de clases igual a la del nodo es:

$$
D_c(k)=L_c(k)-k\pi_c.
$$

La implementación calcula la varianza esperada $V_c(k)$ de esa diferencia y asigna al corte la mayor desviación estandarizada al cuadrado entre las clases:

$$
S_{\max}(k)=\max_c\frac{D_c(k)^2}{V_c(k)},
\qquad
V_c(k)=\frac{k(n-k)}{n-1}\,\pi_c(1-\pi_c).
$$

Elige el corte válido con mayor $S_{\max}(k)$. **Esta puntuación no es un p-valor**: el código no la compara con una distribución nula ni con `alpha`. El p-valor que decide si se puede dividir es el del contraste predictor–clase descrito antes.

La celda siguiente guarda el modelo en `ct_visual` y muestra su tabla, con una fila por nodo. Para interpretarla, fíjate en:

- **Profundidad**: cuántas divisiones hay desde la raíz; **n**: cuántos productos de entrenamiento llegan al nodo.
- **Variable** y **Corte**: la regla usada en un nodo que divide. Por ejemplo, `feat_12 ≤ corte` envía los productos que cumplen la condición por la rama **≤** y los demás por la rama **>**.
- **Clase** y **Terminal**: la clase más frecuente en ese nodo y si es una hoja (`True`) o un nodo interno (`False`). En una hoja, la clase mayoritaria es la que se asigna a los productos que llegan allí.
- **`log10(p)`**: el p-valor ajustado registrado. En una división corresponde al predictor utilizado; si el nodo queda como hoja tras evaluar los predictores, puede mostrar el menor p-valor ajustado calculado allí. Cuanto más negativo es, menor es ese p-valor.

El límite de profundidad facilita interpretar y dibujar el árbol y restringe su complejidad, pero no garantiza el mejor rendimiento predictivo: un árbol demasiado simple puede pasar por alto patrones. Más adelante evaluaremos las predicciones y exploraremos la poda.

In [ ]:
ct_visual = ArbolCondicional(max_depth=3, splitstat="maximum").fit(X_train, y_train)  # Ajusta hasta tres niveles usando solo entrenamiento.
ct_visual.tabla_nodos()  # Muestra una fila por nodo: regla, tamaño, clase y significación.

#### Representación extendida: frecuencias de las clases en las hojas

El dibujo muestra el árbol ajustado con los datos de entrenamiento. Para clasificar un producto, comenzamos en la **raíz** y seguimos las reglas hasta llegar a una **hoja**:

1. Cada nodo interno indica su número, el tamaño `n` y una regla como `feat_12 ≤ corte`. Sigue la rama marcada con **≤** si el producto cumple la regla y la rama **>** si no la cumple.
2. El valor `log₁₀(p)` resume, en escala logarítmica, el p-valor ajustado del contraste asociado a la variable elegida. Cuanto más negativo es, más pequeño es el p-valor; no es una medida de acierto.
3. En cada rectángulo terminal, `n` cuenta los productos de entrenamiento que llegaron a esa hoja. El texto muestra la **clase mayoritaria** y el porcentaje que representa dentro de la hoja. Esa es la clase que el árbol asigna a un producto nuevo que siga el mismo camino.

Las barras pequeñas muestran la **proporción de cada clase dentro de la hoja**; la leyenda inferior relaciona cada color con su clase. Por ejemplo, si una hoja contiene 100 productos de entrenamiento y 70 pertenecen a una misma clase, su proporción es del 70 % y esa será la clase asignada. Una hoja con una barra claramente dominante está más concentrada en una clase; si varias barras tienen alturas parecidas, mezcla más clases.

Estas proporciones describen los productos de **entrenamiento** que llegaron a cada hoja. Un porcentaje alto en una hoja no garantiza el mismo acierto con productos nuevos: para comprobar la capacidad predictiva necesitamos evaluar las predicciones en test.

In [ ]:
dibujar_arbol(  # Llama a la función de gráficos con el árbol que acabamos de ajustar.
    ct_visual,  # Pasa el modelo para que se lean sus nodos y sus reglas.
    profundidad=3,  # Dibuja hasta tres divisiones desde la raíz.
    extendido=True,  # Añade barras con las proporciones de las clases en las hojas.
    titulo="Árbol condicional · representación extendida",  # Escribe un título descriptivo sobre el gráfico.
)

#### Representación simple: reglas y clase asignada

Esta vista muestra las mismas reglas que la representación extendida, pero con menos información visual para que sea más fácil seguir el recorrido de clasificación. Empieza en la raíz y, en cada nodo interno, compara el valor del producto con el corte indicado: si cumple `variable ≤ corte`, sigue la rama **≤**; si no, sigue la rama **>**. Repite el proceso hasta llegar a una hoja.

Cada hoja indica el número de productos de entrenamiento que llegaron a ella (`n`), su **clase mayoritaria** y el porcentaje de esa clase. Esa es la clase que el árbol asigna al producto que llega a esa hoja. El porcentaje resume solo la clase dominante: aquí se omiten las barras que muestran el reparto de **todas** las clases y su leyenda, presentes en la vista extendida.

La simplificación facilita leer las reglas y el camino hasta la predicción, pero no cambia el árbol ni sus predicciones. Como antes, el porcentaje de una hoja se calcula con entrenamiento; la evaluación en test permite comprobar cómo clasifica productos no usados al ajustar el modelo.

In [ ]:
dibujar_arbol(  # Dibuja el mismo árbol en su versión más sencilla.
    ct_visual,  # Utiliza el árbol limitado a tres niveles.
    profundidad=3,  # Conserva el mismo límite de profundidad que en la vista anterior.
    extendido=False,  # Oculta las barras de proporciones para simplificar la lectura.
    titulo="Árbol condicional · reglas de clasificación",  # Identifica qué representa la figura.
)

### Capacidad predictiva con profundidad sin límite explícito

Ahora ajustamos un árbol sin fijar `max_depth`: a diferencia del ejemplo anterior, no imponemos de antemano cuántos niveles puede alcanzar. **Esto no significa que crezca sin ninguna condición de parada.** En cada nodo deja de dividirse si ya contiene una sola clase, si no hay evidencia estadística suficiente de asociación con los predictores o si no encuentra un corte que respete los tamaños mínimos. Con los valores predeterminados, se necesitan al menos 20 observaciones para intentar una división y cada lado debe contener al menos 7 observaciones; el contraste utiliza `alpha=0.05`.

El árbol se ajusta con `X_train` e `y_train`. Después, aplica las reglas aprendidas a `X_test`, que no intervino en el ajuste, y compara las predicciones con las clases reales de test. La matriz de confusión presenta las **predicciones en filas** y las **clases reales en columnas**; la proporción de aciertos resume qué parte de los productos de test clasificó correctamente.

Por último, observamos **Nodos**, **Hojas** y **Profundidad** para describir el tamaño del modelo. Al no tener un límite explícito de profundidad, este árbol puede ser más complejo que el anterior, pero eso no garantiza mayor acierto en test. Compara el resultado predictivo con el tamaño: más reglas pueden captar más detalle y también hacer el modelo menos sencillo de interpretar.

In [ ]:
ct_mod = ArbolCondicional(splitstat="maximum").fit(X_train, y_train)  # Ajusta el árbol sin un límite manual de profundidad.
pred_ct = ct_mod.predict(X_test)  # Predice la clase de cada producto reservado para test.
evaluacion_ct = evaluar_clasificador(y_test, pred_ct, clases)  # Calcula matriz y aciertos con las etiquetas reales.
t_ct = evaluacion_ct.confusion  # Guarda la matriz en una variable para reutilizarla más adelante.
t_ct  # Muestra las predicciones en filas y las clases reales en columnas.
print(f"Proporción global de predicciones correctas {round(evaluacion_ct.acierto * 100, 2)} %")  # Muestra la proporción global de predicciones correctas.

In [ ]:
pd.Series({"Nodos": len(ct_mod.nodos), "Hojas": len(ct_mod.hojas()),  # Cuenta nodos totales y hojas terminales.
           "Profundidad": max(n.profundidad for n in ct_mod.nodos)})  # Encuentra la mayor profundidad alcanzada.

### Capacidad predictiva por clase

El **acierto global** indica qué proporción de todos los productos de test clasificó correctamente. Sin embargo, puede ocultar que el árbol acierta mucho en unas clases y falla con frecuencia en otras, especialmente si las clases tienen tamaños distintos.

La tabla muestra el **acierto dentro de cada clase real**: de los productos que pertenecen realmente a una clase, qué proporción recibió la etiqueta correcta. Para una clase $c$, se calcula como:

$$
\text{Acierto}_c = \frac{\text{productos de clase }c\text{ clasificados correctamente}}{\text{total de productos reales de clase }c}.
$$

En la matriz de confusión, las **predicciones están en las filas** y las **clases reales en las columnas**. Por eso, para calcular el acierto de una clase se divide su casilla diagonal entre el total de su columna. El gráfico representa esos resultados con una barra por clase: valores cercanos a 1 indican más aciertos en esa clase; valores bajos señalan que conviene revisar qué otras clases confunde el árbol.

Compara las barras entre sí y con el acierto global. Una diferencia entre clases describe el comportamiento en este conjunto de test; por sí sola no explica la causa ni garantiza que se repita en otros datos.

In [ ]:
evaluacion_ct.por_clase  # Muestra el acierto dentro de cada clase real.
barras_clases(  # Crea un gráfico de barras para comparar visualmente esos aciertos.
    evaluacion_ct.por_clase,  # Pasa la serie donde cada índice es una clase.
    "Árbol condicional · acierto por clase",  # Añade un título que identifica modelo y medida.
)

**Interpretación del gráfico.** El eje vertical muestra la proporción de productos de cada clase real que el árbol clasificó correctamente; las barras más altas indican mayor acierto para esa clase. En esta partición de test, el árbol obtiene resultados altos para **Class 2, Class 5 y Class 6** (alrededor del 0,88–0,90), y también para **Class 8** (aproximadamente 0,83). **Class 9** queda en un nivel intermedio (cerca de 0,70), mientras que el acierto es menor para **Class 7** (0,45), **Class 1** y **Class 4** (alrededor de 0,27–0,30), y especialmente **Class 3** (cerca de 0,14).

Esto indica que el modelo reconoce unas clases con más frecuencia que otras en este test. No muestra por sí solo con qué clases se confunden las etiquetas con menor acierto; para verlo, consulta la matriz de confusión. Los valores son proporciones calculadas por separado dentro de cada clase real, no el número de aciertos ni el tamaño de cada clase, y describen únicamente esta partición.

### Micropráctica 7.1

Calcula el acierto del árbol para `Class_1` a partir de la matriz `t_ct`, sin volver a ajustar el modelo. Sustituye los dos huecos de la celda siguiente: el primero identifica los aciertos y el segundo, el total de observaciones **reales** de esa clase.

**Para pensar:** ¿coincide con el valor de `evaluacion_ct.por_clase`? ¿Por qué puede ser diferente del acierto global? Las filas son predicciones y las columnas son clases reales. Si no completas los huecos, la celda indicará que la actividad está pendiente y el notebook podrá continuar.

In [ ]:
# Sustituye cada ... por una expresión de la matriz t_ct.
# Pista: t_ct.loc["Class_1", "Class_1"] accede a una casilla;
# t_ct["Class_1"].sum() suma los casos reales de la columna.
aciertos_mp71 = ...
total_real_mp71 = ...

if aciertos_mp71 is ... or total_real_mp71 is ...:
    print("Micropráctica 7.1 pendiente: completa los dos huecos.")
else:
    acierto_mp71 = aciertos_mp71 / total_real_mp71
    display(pd.Series({"Acierto calculado": acierto_mp71,
                       "Acierto por clase del árbol": evaluacion_ct.por_clase.loc["Class_1"],
                       "Acierto global": evaluacion_ct.acierto}))

#### Explorar la representación

El control interactivo solo cambia la **vista del dibujo**, no vuelve a ajustar el modelo ni cambia las predicciones guardadas en `pred_ct`. El deslizador colapsa las ramas más profundas; la casilla decide si mostramos las frecuencias de las clases en las hojas visibles.

En Jupyter, al mover un control se vuelve a llamar a la función que aparece en la celda siguiente. Por eso esa celda define primero qué hacer con los valores seleccionados y después construye el control.

In [ ]:
def explorar_arbol(profundidad, extendido):  # Define qué debe ocurrir cuando cambie el control.
    escala_texto = 1 + 0.18 * max(0, profundidad - 3)  # Aumenta el tamaño de las etiquetas en vistas más profundas.
    dibujar_arbol(  # Actualiza el dibujo usando la configuración que eligió el estudiante.
        ct_mod,  # Pasa el árbol completo ya ajustado, sin volver a entrenarlo.
        profundidad=profundidad,  # Usa el valor actual del deslizador.
        extendido=extendido,  # Usa el estado actual de la casilla de frecuencias.
        titulo=f"Árbol condicional · vista hasta profundidad {profundidad}",  # Incluye el nivel en el título.
        escala_texto=escala_texto,  # Agranda etiquetas y leyenda sin ampliar el lienzo.
    )

control_arbol = widgets.interactive(  # Conecta controles gráficos con la función anterior.
    explorar_arbol,  # Indica qué función se ejecutará al cambiar valores.
    profundidad=widgets.IntSlider(value=3, min=1, max=5, description="Profundidad", continuous_update=False),  # Crea un deslizador entero.
    extendido=widgets.Checkbox(value=True, description="Frecuencias en hojas"),  # Crea una casilla activada inicialmente.
)
display(control_arbol)  # Muestra el deslizador y la casilla.

#### Inspeccionar un nodo

El árbol guarda en cada nodo los productos de **entrenamiento** que han llegado hasta él. Usa el deslizador para elegir el número del nodo; coincide con la etiqueta que aparece en el dibujo. El widget reconstruye, en orden, las reglas que se siguen desde la raíz hasta ese nodo y muestra cuál rama se toma en cada paso. Después presenta la información del nodo seleccionado, una tabla con las frecuencias y proporciones por clase, y un **gráfico de barras horizontal** para comparar las proporciones de un vistazo. Los colores de las barras coinciden con los colores de las clases en el árbol.

- **Recorrido:** cada fila indica el nodo padre, la regla aplicada y el nodo al que conduce. La condición `variable ≤ corte` lleva a la rama izquierda; `variable > corte`, a la derecha. Si seleccionas la raíz, no hay reglas previas.
- **Regla y tamaño:** en un nodo interno, la variable y el corte indican cómo se separan los productos; `n` es cuántos llegaron a él. En una hoja ya no hay otra regla que seguir.
- **Frecuencia:** número de productos de cada clase dentro del nodo.
- **Proporción:** frecuencia de cada clase dividida por `n`; las proporciones suman 1. Describe la composición de ese grupo de entrenamiento, no la probabilidad de que un producto nuevo pertenezca a cada clase.

El nodo raíz reúne todos los productos de entrenamiento. En los nodos siguientes se incluyen solo los que cumplieron las reglas del recorrido hasta allí. Aunque la tabla indica la clase mayoritaria de cada nodo, para clasificar un producto el árbol sigue las reglas hasta una hoja; la clase mayoritaria de esa hoja es la predicción final.

In [ ]:
control_nodo = widgets.interactive(  # Crea el widget que actualizará la inspección al elegir otro nodo.
    inspeccionar_nodo,  # Usa la función reutilizable definida en s7_graficos.py.
    modelo=widgets.fixed(ct_mod),  # Mantiene fijo el árbol completo que se va a inspeccionar.
    nodo=widgets.IntSlider(value=0, min=0, max=len(ct_mod.nodos)-1, description="Nodo", continuous_update=False),  # Permite seleccionar cualquiera de sus nodos.
)
display(control_nodo)  # Muestra el deslizador y la inspección dinámica.

### Poda manual por significación

La **poda** simplifica un árbol ya construido: convierte un nodo interno en una hoja y deja de usar sus divisiones descendientes. Las observaciones de entrenamiento que llegaron a ese nodo se resumen por su clase mayoritaria; por tanto, cualquier producto que alcance la hoja podada recibirá esa clase. Podar reduce el número de reglas, aunque también puede eliminar distinciones útiles.

Para esta demostración ajustamos otro árbol con los datos de entrenamiento y usamos `splitstat="quadratic"`. Este parámetro **no realiza el contraste de significación**: puntúa los posibles cortes de un predictor una vez seleccionado el predictor para dividir. Para entender la puntuación, imagina ordenar los productos por el valor de ese predictor y probar un corte después de las primeras $k$ observaciones. Si $L_c(k)$ es el número de productos de clase $c$ a la izquierda del corte y $\pi_c$ es la proporción de esa clase en el nodo, la diferencia respecto a lo que esperaríamos si ambos lados conservaran la mezcla de clases del nodo es:

$$
D_c(k)=L_c(k)-k\pi_c.
$$

La puntuación cuadrática combina las diferencias de **todas** las clases:

$$
Q(k)=\frac{n-1}{k(n-k)}\sum_{c=1}^{C}\frac{D_c(k)^2}{\pi_c},
$$

donde $n$ es el número de productos que llegan al nodo. El algoritmo elige, entre los cortes que respetan los tamaños mínimos, el que obtiene mayor $Q(k)$. Cuanto mayor sea la puntuación, mayor será la separación conjunta de las clases respecto a la mezcla esperada si el corte no las separara.

**Comparación con `splitstat="maximum"`:** ambos criterios puntúan cortes, pero de manera distinta. `"maximum"` toma, para cada corte, la mayor desviación estandarizada entre las clases; por eso una clase muy separada puede dominar la puntuación. `"quadratic"` suma las desviaciones cuadráticas de las clases presentes, de modo que varias diferencias más moderadas también pueden contribuir al resultado. Ninguno de estos dos puntajes es por sí mismo un p-valor ni el contraste que decide si se divide: el p-valor de la tabla procede del contraste predictor–clase anterior.

La tabla muestra, para cada nodo interno, el predictor utilizado y su `log10(p)`: el p-valor ajustado del contraste de selección del predictor. Un valor más negativo representa un p-valor menor y mayor evidencia contra la hipótesis nula de que el predictor no está asociado con la clase en ese nodo.

Después aplicaremos una regla manual: marcar para poda las divisiones cuyo `log10(p)` supere un umbral elegido. Al podar un nodo se ignora todo su subárbol y se predice usando la distribución de clases de ese nodo. El umbral de esta demostración es didáctico, no universal: sirve para observar cómo cambia la complejidad y el resultado del árbol, no para afirmar que exista un valor óptimo para cualquier problema. La evaluación en test permite describir el efecto de la regla; no conviene ajustar repetidamente el umbral para escoger un modelo usando ese mismo test.

In [ ]:
ct_poda = ArbolCondicional(splitstat="quadratic").fit(X_train, y_train)  # Ajusta el árbol con datos de entrenamiento y puntuación cuadrática para elegir cortes.
pd.Series({"Nodos": len(ct_poda.nodos), "Hojas": len(ct_poda.hojas())})  # Resume el tamaño del árbol antes de podarlo.

#### Revisar los nodos internos y sus p-valores

La tabla siguiente incluye solo los nodos que efectivamente dividen el árbol. Los p-valores son los ajustados del contraste predictor–clase en cada nodo; se muestran como `log10(p)` para poder representar valores muy pequeños. **Cuanto más negativo es `log10(p)`, menor es el p-valor y mayor la evidencia contra la hipótesis nula.** La puntuación `quadratic` eligió el corte; no es el p-valor de la tabla. Ordenaremos los nodos de menor a mayor `log10(p)` para ver primero los p-valores más pequeños.

In [ ]:
nodos_ct = ct_poda.tabla_nodos()  # Convierte todos los nodos en una tabla legible.
internos_ct = nodos_ct.loc[~nodos_ct["Terminal"]].copy()  # Conserva solo los nodos que tienen una división.
internos_ct[["Profundidad", "n", "Variable", "Corte", "log10(p)"]].sort_values("log10(p)")  # Ordena desde el p-valor más pequeño.

#### Resumir la distribución de los p-valores

Los cuantiles condensan los p-valores de todos los nodos internos. El percentil **0 %** es el valor mínimo (el p-valor más pequeño); el **50 %** es la mediana; y el **100 %** es el máximo (el p-valor más grande). Como se resume `log10(p)`, un cuantil más negativo corresponde a p-valores más pequeños. Este resumen describe la evidencia registrada en los nodos; por sí solo no determina qué umbral de poda conviene usar.

In [ ]:
cuantiles_logp_ct = internos_ct["log10(p)"].quantile(np.arange(0, 1.01, 0.1))  # Calcula cuantiles desde el mínimo hasta el máximo.
cuantiles_logp_ct.index = [f"{int(percentil * 100)} %" for percentil in cuantiles_logp_ct.index]  # Etiqueta cada fila como percentil.
cuantiles_logp_ct.rename("log10(p)").to_frame()  # Presenta el resumen en una tabla compacta.

Usaremos `log10(p) > −95` como regla **puramente ilustrativa** para explorar la poda. Como el logaritmo es creciente, un valor más cercano a cero significa un p-valor ajustado mayor y evidencia menos fuerte; así, la regla convierte en hojas los nodos con `log10(p) > −95` y conserva sus divisiones solo cuando `log10(p) ≤ −95` (equivale a $p_{\text{aj}}\le 10^{-95}$).

El umbral es deliberadamente muy exigente: no equivale al nivel habitual `alpha=0.05` del contraste de crecimiento, sino que se eligió para que se aprecie cómo se simplifica este árbol. En el resumen anterior, el percentil 10 de `log10(p)` es aproximadamente −82; por ello, −95 conserva solo una fracción pequeña de las divisiones y poda la mayoría en esta muestra. No es un valor universal ni un criterio recomendado para otros datos.

In [ ]:
ids_poda = internos_ct.index[internos_ct["log10(p)"] > -95].tolist()  # Recoge los nodos internos que cumplen el umbral.
pred_ct_poda = ct_poda.predict(X_test, podados=ids_poda)  # Predice tratando esos nodos como hojas terminales.
evaluacion_ct_poda = evaluar_clasificador(y_test, pred_ct_poda, clases)  # Compara predicciones con clases reales.
t_ct_poda = evaluacion_ct_poda.confusion  # Guarda la matriz de confusión de la versión podada.
t_ct_poda  # Muestra la matriz para ver qué clases acierta o confunde.
print(f"Proporción global de predicciones correctas {round(evaluacion_ct_poda.acierto * 100, 2)} %")  # Muestra la fracción de predicciones de test que son correctas.
pd.Series({"Hojas tras la poda": len(ct_poda.hojas(ids_poda)), "Predicciones ausentes": pred_ct_poda.isna().sum()})  # Comprueba tamaño y cobertura.

El deslizador de la siguiente celda cambia el umbral de poda. Para cada valor se vuelven a calcular las predicciones, la matriz y el número de hojas; así comparamos el compromiso entre sencillez y acierto sin modificar los datos.

In [ ]:
def explorar_poda_condicional(log10_umbral):  # Define la actualización del modelo para un umbral elegido.
    podados = internos_ct.index[internos_ct["log10(p)"] > log10_umbral].tolist()  # Busca qué nodos cumplen el nuevo umbral.
    prediccion = ct_poda.predict(X_test, podados=podados)  # Predice test con los nodos seleccionados como hojas.
    resultado = evaluar_clasificador(y_test, prediccion, clases)  # Calcula acierto y matriz para esa predicción.
    display(pd.Series({"Hojas": len(ct_poda.hojas(podados)), "Proporción de aciertos": resultado.acierto}))  # Resume sencillez y acierto.
    display(resultado.confusion)  # Muestra el detalle de clases reales y predichas.
    dibujar_arbol(  # Actualiza el dibujo para ver qué ramas se han eliminado.
        ct_poda,  # Pasa el árbol original antes de la poda.
        profundidad=3,  # Limita el dibujo a los tres primeros niveles.
        extendido=False,  # Omite las barras para que el dibujo sea más compacto.
        podados=podados,  # Indica qué nodos se tratan como hojas.
        titulo=f"Poda por log₁₀(p) > {log10_umbral}",  # Escribe el umbral actual en el título.
    )

control_poda = widgets.interactive(  # Vincula el deslizador con la exploración de poda.
    explorar_poda_condicional,  # Ejecuta la función al cambiar el valor.
    log10_umbral=widgets.IntSlider(value=-95, min=-400, max=0, step=3,  # Ofrece umbrales entre −200 y 0.
                                  description="log₁₀(p)", continuous_update=False),  # Actualiza al soltar el deslizador.
)
display(control_poda)  # Muestra el control y sus resultados.

#### La paradoja: más sobreajuste, pero también más acierto en test

Al acercar el umbral a cero se podan menos nodos y el árbol conserva más divisiones. En esta partición, al pasar de −95 a 0, el acierto de entrenamiento sube de **64,22 %** a **72,54 %**, y el de test de **63,93 %** a **70,09 %**. No es una contradicción: el sobreajuste no significa que el acierto de test tenga que bajar cada vez que el modelo se hace más complejo. Se aprecia aquí en que la brecha entre entrenamiento y test aumenta de **0,28** a **2,46 puntos porcentuales**: el árbol sin podar mejora en ambos conjuntos, pero mejora más en entrenamiento.

En esta partición, las divisiones adicionales parecen aportar suficiente información útil para que el acierto de test también suba; al mismo tiempo, la brecha mayor indica que parte de la mejora está más ligada a los datos de entrenamiento y podría no repetirse en otros productos. **Podar no siempre mejora el acierto**: puede eliminar divisiones que capturan patrones reales y agrupar en una hoja productos de clases distintas, lo que aumenta el sesgo y puede llevar a un árbol demasiado simple. La poda resulta útil cuando la reducción de variabilidad compensa esa pérdida de información, no por el mero hecho de simplificar.

Por eso no se debe concluir ni que el árbol sin podar sea siempre mejor ni que haya que podar siempre. El deslizador consulta test para cada umbral, así que úsalo para observar el efecto, no para escoger el mejor valor; la complejidad debe seleccionarse con validación cruzada sobre entrenamiento y test debe reservarse para la evaluación final.

### Poda por coste-complejidad

Como alternativa a la regla manual anterior, construiremos una secuencia de árboles cada vez más pequeños y compararemos su error mediante validación cruzada. El árbol inicial usa `DecisionTreeClassifier` de scikit-learn con **entropía** para elegir las divisiones.

#### 1. Limitar el crecimiento inicial

- `min_samples_split=10`: se necesitan al menos 10 observaciones en un nodo para intentar dividirlo.
- `min_samples_leaf=5`: cada hijo resultante debe contener al menos 5 observaciones.
- `max_leaf_nodes=250`: limita el árbol inicial a un máximo de 250 hojas.
- Además, el código auxiliar descarta divisiones muy pequeñas: conserva un nodo si el producto de sus observaciones por su impureza alcanza al menos el **0,05 %** del valor correspondiente a la raíz.

Estos límites controlan el crecimiento del árbol base. En esta muestra, el árbol sin poda alcanza las 250 hojas; la trayectoria de poda parte de ese modelo y genera versiones progresivamente más simples.

#### 2. Penalizar el tamaño del árbol

Para cada subárbol se combina el número de errores de clasificación en entrenamiento con una penalización por sus hojas:

$$
C_{\alpha}(T)=R_{\text{train}}(T)+\alpha\,|L(T)|,
$$

donde $R_{\text{train}}(T)$ es el número de errores que comete el árbol en entrenamiento y $|L(T)|$ es su número de hojas. Al aumentar $\alpha$, tener muchas hojas resulta más costoso y se prefieren árboles más pequeños.

En cada paso se poda la rama cuya eliminación aumenta menos los errores por cada hoja que desaparece. Para un nodo $t$ y su subárbol $T_t$, el coste de esa poda es:

$$
\alpha_t=\frac{R_{\text{train}}(t)-R_{\text{train}}(T_t)}{|L(T_t)|-1}.
$$

Aquí, $R_{\text{train}}(t)$ cuenta los errores si toda la rama se sustituye por una sola hoja, y $R_{\text{train}}(T_t)$ los errores que comete la rama antes de podarla. El procedimiento repite la eliminación de las ramas menos costosas y guarda los árboles resultantes; así se puede comparar la complejidad sin volver a ajustar manualmente cada tamaño.

#### 3. Elegir el tamaño con validación cruzada

No elegiremos el árbol mirando cuál obtiene más aciertos en test. Más adelante dividiremos **solo el entrenamiento** en diez pliegues, compararemos el error de los distintos tamaños y usaremos test para la evaluación final. La validación cruzada ayuda a elegir entre estos candidatos, aunque el tamaño ganador todavía puede variar con otros datos.

In [ ]:
tr = arbol_entropia(X_train, y_train, seed=SEED)  # Ajusta con entropía usando únicamente las filas de entrenamiento.
camino_poda = trayectoria_poda(tr)  # Genera la secuencia de subárboles, del más grande al más pequeño.

# Resume la trayectoria en una tabla legible; se omite la lista interna de nodos podados.
error_train_pct = 100 * camino_poda["Errores train"] / len(y_train)
tabla_poda = pd.DataFrame(
    {
        "α de poda": camino_poda["alpha"].map(
            lambda valor: "Sin poda" if not np.isfinite(valor) else f"{valor:.2f}".replace(".", ",")
        ).to_numpy(),
        "Errores en entrenamiento": camino_poda["Errores train"].map(
            lambda valor: f"{int(valor):,}".replace(",", ".")
        ).to_numpy(),
        "Error train (%)": error_train_pct.map(
            lambda valor: f"{valor:.2f}".replace(".", ",") + " %"
        ).to_numpy(),
    },
    index=pd.Index(camino_poda["Hojas"].to_numpy(), name="Hojas"),
)
indices_resumen = np.unique(np.linspace(0, len(tabla_poda) - 1, num=min(10, len(tabla_poda)), dtype=int))  # Elige hasta diez tamaños representativos.
print(f"Mostrando {len(indices_resumen)} de {len(tabla_poda)} subárboles; el recorrido completo queda en camino_poda.")  # Explica el resumen.
display(tabla_poda.iloc[indices_resumen])  # Presenta una tabla breve junto al gráfico completo.

# Visualiza cómo cambia el error de entrenamiento según el tamaño del árbol.
orden_poda = camino_poda.sort_values("Hojas")
fig, ax = plt.subplots()
_ = ax.plot(
    orden_poda["Hojas"],
    100 * orden_poda["Errores train"] / len(y_train),
    "o-",
    markersize=4,
)
_ = ax.set(
    xlabel="Número de hojas",
    ylabel="Error de entrenamiento (%)",
    title="Árbol con más hojas: menor error en entrenamiento",
)
ax.grid(alpha=0.2)
plt.show()

#### Qué significan los errores de entrenamiento

- **Errores en entrenamiento**: número de productos de `X_train` que ese subárbol clasifica mal.
- **Error train (%)**: esos errores expresados como porcentaje del total de productos de entrenamiento:

$$
\text{Error train (\%)}=100\times\frac{\text{errores en entrenamiento}}{\text{número de productos de entrenamiento}}.
$$

Por ejemplo, con **250 hojas**, el árbol comete **8 394 errores**, equivalentes al **27,81 %** del entrenamiento. Al podar y reducir las hojas, el error de entrenamiento suele subir porque el árbol dispone de menos reglas para ajustar esos mismos datos. Esta medida describe el ajuste en entrenamiento, no el rendimiento en productos nuevos; para comparar la capacidad de generalización usamos el error de validación cruzada de la sección siguiente.

#### Validación cruzada de la poda

Para comparar los tamaños sin consultar test, dividimos **solo el entrenamiento** en diez pliegues. En cada vuelta se ajusta un árbol con nueve pliegues y se calcula su error de clasificación en el décimo; se repite para los tamaños de la secuencia y se promedian los errores.

La tabla resume candidatos representativos: **menor error CV es mejor**, mientras que la desviación estándar indica cuánto varía el error entre pliegues. Para el árbol de **250 hojas**, el error medio es **30,20 %** y la DE entre pliegues **0,44 puntos porcentuales**: describe cuánto varían los diez errores de pliegue alrededor de la media, no un intervalo de confianza. El gráfico muestra la trayectoria completa y esa variación. Test queda reservado para evaluar el modelo después de escoger su tamaño; el mínimo observado en CV es una estimación, no una garantía de que el mismo árbol sea mejor en otros datos.

**¿Por qué no elegir el tamaño con la regla del codo?** El codo es una pauta visual útil para detectar rendimientos decrecientes, pero su ubicación suele ser subjetiva y no mide directamente el rendimiento predictivo. Aquí el error CV disminuye al conservar más hojas y se aplana gradualmente, sin un quiebre único claro. Su mínimo observado es **30,17 % con 215 hojas**. Por eso comparamos el error CV directamente en vez de decidir por un codo ambiguo. Si se quiere priorizar un árbol más pequeño cuando varios errores son parecidos, puede usarse la regla de un error estándar, calculando el error estándar (no solo la desviación estándar mostrada).

In [ ]:
with threadpool_limits(limits=1):  # Limita el paralelismo numérico durante la validación cruzada.
    cv_tr = validar_poda(X_train, y_train, camino_poda, cv=10, seed=SEED)  # Evalúa cada poda en diez particiones.

# Resume diez tamaños representativos; cv_tr conserva todos los candidatos para análisis y ejercicios.
indices_cv = np.unique(np.linspace(0, len(cv_tr) - 1, num=min(10, len(cv_tr)), dtype=int))
tabla_cv = cv_tr.iloc[indices_cv][["Hojas", "alpha", "Error CV", "Desviación CV"]].copy()
tabla_cv = tabla_cv.rename(
    columns={"alpha": "α de poda", "Error CV": "Error CV medio", "Desviación CV": "DE entre pliegues"}
)
tabla_cv["α de poda"] = tabla_cv["α de poda"].map(
    lambda valor: "Sin poda" if not np.isfinite(valor) else f"{valor:.2f}".replace(".", ",")
)
for columna in ["Error CV medio", "DE entre pliegues"]:
    tabla_cv[columna] = tabla_cv[columna].map(
        lambda valor: f"{100 * valor:.2f}".replace(".", ",") + " %"
    )

candidatos_mejor_cv = cv_tr.loc[cv_tr["Error CV"] == cv_tr["Error CV"].min()]
hojas_mejor_cv = ", ".join(candidatos_mejor_cv["Hojas"].astype(int).astype(str))
print(
    f"Menor error CV medio: {100 * cv_tr['Error CV'].min():.2f} % "
    f"({len(candidatos_mejor_cv)} candidatos empatados: {hojas_mejor_cv} hojas)."
)
print(f"Mostrando {len(tabla_cv)} de {len(cv_tr)} tamaños; la tabla completa queda en cv_tr.")
display(tabla_cv.set_index("Hojas"))

In [ ]:
orden_cv = cv_tr.sort_values("Hojas")  # Ordena de menor a mayor complejidad para leer mejor el eje horizontal.
error_cv = 100 * orden_cv["Error CV"]  # Expresa la media de error como porcentaje.
desviacion_cv = 100 * orden_cv["Desviación CV"]  # Expresa la variabilidad entre pliegues en la misma escala.
mejores_cv = cv_tr.loc[cv_tr["Error CV"] == cv_tr["Error CV"].min()]  # Identifica los candidatos con menor error medio.

fig, ax = plt.subplots(figsize=(8, 4.5))  # Da espacio suficiente para las etiquetas y la leyenda.
_ = ax.errorbar(
    orden_cv["Hojas"],
    error_cv,
    yerr=desviacion_cv,
    fmt="o-",
    markersize=4,
    capsize=3,
    linewidth=1.5,
    label="Error medio ± 1 DE entre pliegues",
)
_ = ax.scatter(
    mejores_cv["Hojas"],
    100 * mejores_cv["Error CV"],
    marker="*",
    s=150,
    color="darkorange",
    zorder=3,
    label="Menor error CV medio",
)
_ = ax.set(
    xlabel="Número de hojas",
    ylabel="Error de clasificación CV (%)",
    title="Validación cruzada: error según el tamaño del árbol",
)
ax.grid(alpha=0.25)
_ = ax.legend(frameon=False)
fig.tight_layout()
plt.show()

### Micropráctica 7.2 — Seleccionar una poda con validación cruzada (opcional)

Usa la tabla `cv_tr` para localizar el tamaño de árbol con menor error medio de validación cruzada. Sustituye el hueco por el nombre exacto de la columna que contiene ese error; puedes consultar los nombres en la tabla anterior.

**Para pensar:** ¿cuántas hojas tiene el candidato? ¿Por qué se compara con validación cruzada de entrenamiento y no escogiendo el tamaño que mejor acierta en test? Un mínimo observado no garantiza que el mismo tamaño gane en otra muestra.

In [ ]:
# Sustituye ... por el nombre de la columna de error medio, entre comillas.
columna_error_mp72 = ...

if columna_error_mp72 is ...:
    print("Micropráctica 7.2 pendiente: completa el nombre de la columna.")
else:
    minimo_mp72 = cv_tr[columna_error_mp72].min()
    candidatos_mp72 = cv_tr.loc[cv_tr[columna_error_mp72] == minimo_mp72]
    display(candidatos_mp72[["Hojas", columna_error_mp72]])  # Muestra también posibles empates.

#### Modelo podado a diez hojas

Seleccionamos de la secuencia el árbol con al menos diez hojas más cercano a ese tamaño. Puede que la secuencia no tenga exactamente diez hojas. Evaluamos este modelo en test y dibujamos el árbol entero para interpretar las reglas que conserva.

In [ ]:
poda_10 = podar_a_hojas(camino_poda, numero=10)  # Recupera los nodos que se deben podar para conservar alrededor de diez hojas.
pred_poda_10 = tr.predict(X_test, podados=poda_10)  # Predice test con el árbol simplificado.
evaluacion_poda_10 = evaluar_clasificador(y_test, pred_poda_10, clases)  # Calcula las métricas de test.
evaluacion_poda_10.confusion  # Muestra la matriz de confusión del árbol podado.
print(f"Proporción global de predicciones correctas {round(evaluacion_poda_10.acierto * 100, 2)} %")  # Muestra el acierto global en test.
len(tr.hojas(poda_10))  # Cuenta cuántas hojas tiene la versión podada.
profundidad_poda_10 = max(tr.nodos[i].profundidad for i in tr.hojas(poda_10))  # Encuentra su hoja más profunda.
dibujar_arbol(tr, profundidad=profundidad_poda_10, podados=poda_10, titulo="Árbol podado a diez hojas")  # Dibuja todas las hojas conservadas.

#### Explorar tamaños de árbol en test

El menú ofrece los tamaños de hoja disponibles en la secuencia de poda. Al elegir uno, **no se vuelve a entrenar el árbol**: se colapsan las ramas necesarias del árbol ajustado con `X_train` y se predicen las mismas filas de `X_test`. El control muestra el número de hojas, la proporción de aciertos, la matriz de confusión y los tres primeros niveles del árbol para comparar su complejidad y sus predicciones.

> **Importante:** cada opción se evalúa en `X_test`. Usa el control para explorar cómo cambia el resultado, no para escoger el tamaño que mejor sale en test; esa elección debe basarse en la validación cruzada de entrenamiento. Test se reserva para una evaluación final.

In [ ]:
def explorar_poda_coste(hojas):  # Define la actualización cuando se elige otro tamaño de árbol.
    podados = podar_a_hojas(camino_poda, numero=hojas)  # Obtiene los nodos a eliminar para ese tamaño.
    prediccion = tr.predict(X_test, podados=podados)  # Predice test con las ramas restantes.
    resultado = evaluar_clasificador(y_test, prediccion, clases)  # Calcula matriz y acierto.
    display(pd.Series({"Hojas": len(tr.hojas(podados)), "Proporción de aciertos": resultado.acierto}))  # Muestra el resumen numérico.
    display(resultado.confusion)  # Permite observar los aciertos y errores por clase.
    dibujar_arbol(tr, profundidad=3, podados=podados, titulo=f"Árbol con {hojas} hojas · primeros niveles")  # Dibuja la parte superior.

control_poda_coste = widgets.interactive(  # Conecta un menú desplegable con la función anterior.
    explorar_poda_coste,  # Indica qué función actualizará los resultados.
    hojas=widgets.Dropdown(options=sorted(camino_poda["Hojas"].unique()),  # Ofrece los tamaños de hojas disponibles.
                           value=len(tr.hojas(poda_10)), description="Hojas"),  # Selecciona por defecto el árbol de diez hojas.
)
display(control_poda_coste)  # Muestra el menú y el árbol correspondiente.

## Ejercicio 7-1 — Explorar un árbol condicional

**Objetivo:** investigar cómo cambia un árbol al modificar **un solo parámetro** de `ArbolCondicional`. Escoge uno entre `max_depth`, `min_samples_split`, `min_samples_leaf` y `alpha`; decide tú qué valores comparar y justifica por qué son razonables. Incluye el valor predeterminado como referencia.

Mantén fija la partición: ajusta cada modelo con `X_train`, `y_train` y evalúa las mismas filas de `X_test`. Registra en una tabla el parámetro y su valor, el número de hojas, la profundidad máxima y el acierto en entrenamiento y test.

**Cuidado:** el test sirve aquí para comparar los resultados de forma descriptiva, no para escoger el valor que más acierta. Para seleccionar un modelo, utiliza validación cruzada sobre entrenamiento y reserva test para la evaluación final.

In [ ]:
# Pistas para plantear el experimento (sin una solución completa):
#
# 1) Antes de programar, anota:
#    - el único parámetro que vas a cambiar;
#    - qué controla y qué esperas que ocurra al aumentarlo o reducirlo;
#    - varios valores que permitan ver una tendencia, incluyendo el predeterminado.
#
# 2) Construye un ArbolCondicional nuevo por cada valor y ajústalo solo con
#    X_train e y_train. Mantén los otros argumentos en su valor predeterminado.
#    Ejemplo de la forma del constructor: ArbolCondicional(max_depth=...).fit(...)
#
# 3) Para cada modelo, usa .predict(X_train) y .predict(X_test). La función
#    evaluar_clasificador(...).acierto devuelve el acierto; modelo.hojas() permite
#    contar hojas, y los atributos de modelo.nodos permiten hallar la profundidad.
#
# 4) Organiza una fila por valor del parámetro. Además de las métricas, registra
#    el tamaño del árbol; así podrás distinguir un cambio de acierto de un cambio
#    de complejidad.
#
# Conceptos que pueden ayudarte:
# - max_depth limita las divisiones en cada camino desde la raíz.
# - min_samples_split fija cuántas observaciones hacen falta para intentar dividir.
# - min_samples_leaf fija el mínimo de observaciones que puede quedar en cada hijo.
# - alpha establece el nivel de significación exigido para permitir una división.
#
# No cambies la partición entre pruebas ni uses test para escoger el ganador.
# En la interpretación, compara entrenamiento con test y comenta si la tendencia
# observada es compatible con tu hipótesis.

## 7.2. Random forest

Un **random forest** ajusta muchos árboles y combina sus predicciones por **voto mayoritario**. La idea es que los árboles aporten información distinta: cada uno aprende de una remuestra de productos y, en cada división, considera solo parte de las características. Al agregar sus votos, el bosque suele ser más estable que un árbol individual, aunque no hay garantía de que siempre acierte más.

### Tres mecanismos del bosque

- **Remuestreo *bootstrap* de productos:** cada árbol toma, con reemplazo, una muestra del conjunto de entrenamiento. Algunas filas pueden repetirse y otras quedar fuera; estas últimas permiten calcular el error **OOB** (*out-of-bag*).
- **Selección aleatoria de características:** en cada división se consideran solo algunos predictores candidatos. Esto ayuda a que los árboles no repitan exactamente las mismas divisiones.
- **Agregación de votos:** cada árbol vota por una clase y el bosque asigna la más votada.

### Primera configuración

Partimos de **50 árboles**, con `bootstrap=True` y `min_samples_leaf=1` (no se exige que una hoja tenga más de una observación). Para cada división se consideran **9 predictores candidatos**: en clasificación, una regla de partida habitual es $\lfloor\sqrt{p}\rfloor$; aquí hay 93 predictores y $\lfloor\sqrt{93}\rfloor=9$. En scikit-learn, esta cantidad se configura con `max_features`.

Estos valores sirven como referencia didáctica, no como una configuración óptima. Después examinaremos el error OOB, la importancia de las características y el efecto de variar el número de predictores candidatos. La evaluación en test sirve para valorar el modelo, no para escoger sus parámetros.

Las funciones de diagnóstico y exploración están en `s7_bosques.py`; aquí ajustaremos el bosque y leeremos sus resultados.

In [ ]:
rf_mod = RandomForestClassifier(  # Crea el bosque con la configuración inicial del ejemplo.
    n_estimators=50,  # Combina 50 árboles.
    max_features=int(np.sqrt(len(predictores))),  # Considera 9 predictores candidatos en cada división.
    bootstrap=True,  # Ajusta cada árbol con una remuestra bootstrap de las filas.
    min_samples_leaf=1,  # No exige más de una observación por hoja.
    random_state=SEED,  # Fija la semilla para repetir el ajuste.
    n_jobs=1,  # Usa un proceso de trabajo.
)
rf_mod.fit(X_train, y_train)  # Ajusta el bosque usando solamente las filas de entrenamiento.

In [ ]:
pred_rf = rf_mod.predict(X_test)  # Predice la clase de cada fila de test con los votos de los árboles.
evaluacion_rf = evaluar_clasificador(y_test, pred_rf, clases)  # Compara esas predicciones con las clases reales.
t_rf = evaluacion_rf.confusion  # Conserva la matriz para volver a usarla en gráficos y comparaciones.
t_rf  # Muestra el detalle de predicciones frente a valores reales.
print(f"Proporción global de predicciones correctas {round(evaluacion_rf.acierto * 100, 2)} %")  # Muestra la fracción global de productos de test bien clasificados.

### Error fuera de bolsa: OOB

Cada árbol se ajusta con una muestra *bootstrap* y deja fuera algunas filas de entrenamiento. Para esas filas, ese árbol puede hacer predicciones sin haberlas usado para aprender. Al reunir los votos de los árboles que dejaron una fila fuera, obtenemos su predicción **OOB** (*out-of-bag*), una evaluación interna al entrenamiento.

La tabla de la siguiente celda sigue cómo cambia el error al añadir árboles. Muestra unos pocos puntos repartidos desde el comienzo hasta el bosque completo: el **error OOB global**, el error dentro de cada clase y la **cobertura**, es decir, la proporción de filas que ya recibió al menos un voto OOB. El cambio entre puntos consecutivos ayuda a ver si el error se reduce, aumenta o se estabiliza; pequeñas oscilaciones son normales.

OOB permite observar la evolución sin consultar test, pero sigue siendo una estimación basada en entrenamiento. Conservamos test para valorar el modelo una vez terminado el análisis, y no interpretamos OOB como una garantía de rendimiento futuro.

[Descripción de random forests y OOB en scikit-learn](https://scikit-learn.org/stable/ensemble.html#random-forests).

In [ ]:
oob_rf = diagnostico_oob(rf_mod, X_train, y_train)  # Acumula predicciones OOB conforme se añaden árboles.

# Cada punto corresponde al bosque formado por los primeros k árboles. Para ver
# la evolución completa sin listar las 50 filas, elegimos hasta seis checkpoints
# aproximadamente equidistantes, incluyendo el primero y el último.
curva_oob = oob_rf.curva
pasos_oob = np.unique(
    np.linspace(1, len(curva_oob), num=min(6, len(curva_oob)), dtype=int)
)
oob_puntos = curva_oob.loc[pasos_oob].copy()

# El error global se calcula sobre las filas que ya tienen al menos un voto OOB.
# Los errores por clase usan solo las filas disponibles de esa clase.
oob_puntos = oob_puntos.rename(
    columns={"OOB": "Error OOB global", "Cobertura": "Cobertura OOB"}
)
oob_puntos["Cambio del error global (pp)"] = 100 * oob_puntos["Error OOB global"].diff()
oob_puntos["Error OOB global"] *= 100
oob_puntos["Cobertura OOB"] *= 100
oob_puntos = oob_puntos.rename(
    columns={clase: f"Error OOB · {clase}" for clase in clases}
)
columnas_error_clase = [f"Error OOB · {clase}" for clase in clases]
oob_puntos[columnas_error_clase] *= 100
columnas_oob = (
    ["Error OOB global", "Cambio del error global (pp)", "Cobertura OOB"]
    + columnas_error_clase
)
tabla_oob_puntos = oob_puntos[columnas_oob].copy()
for columna in columnas_oob:
    if columna == "Cambio del error global (pp)":
        tabla_oob_puntos[columna] = tabla_oob_puntos[columna].map(
            lambda valor: "—" if pd.isna(valor) else f"{valor:+.2f} pp"
        )
    else:
        tabla_oob_puntos[columna] = tabla_oob_puntos[columna].map(
            lambda valor: f"{valor:.1f} %"
        )
display(tabla_oob_puntos)

# Resume el resultado final tras incorporar todos los árboles.
resumen_oob = pd.Series(
    {
        "Árboles": len(rf_mod.estimators_),
        "Características candidatas por división (max_features)": rf_mod.max_features,
        "Error OOB global": f"{100 * curva_oob['OOB'].iloc[-1]:.2f} %",
        "Cobertura OOB": f"{100 * curva_oob['Cobertura'].iloc[-1]:.1f} %",
    },
    name="Resultado final",
)
display(resumen_oob.to_frame())

#### Comparar errores de test y OOB

Comparamos el **error de clasificación** del bosque en dos evaluaciones: `X_test`, que se reservó aparte, y OOB, que se calcula con las filas de entrenamiento que cada árbol dejó fuera de su remuestra. Para que la comparación sea equivalente, el error por clase es la proporción de productos de esa clase que se clasificaron mal; **Global** resume todas las clases.

La tabla mostrará cada error como porcentaje y añadirá **Test − OOB**, en puntos porcentuales. Una diferencia positiva significa que el error observado en test fue mayor que el OOB; una negativa, que fue menor. La diferencia es descriptiva: no es una prueba estadística ni garantiza que ambos errores coincidan en otros datos.

In [ ]:
# Error de test: uno menos el acierto global y el acierto dentro de cada clase.
# Así, los valores quedan en la misma escala que el diagnóstico OOB.
errores_test = pd.concat(
    [
        pd.Series({"Global": 1 - evaluacion_rf.acierto}),
        1 - evaluacion_rf.por_clase,
    ]
)

# Toma el último punto OOB (el bosque completo) y alinea las clases con las de test.
errores_oob = oob_rf.curva.iloc[-1][["OOB"] + list(clases)].copy()
errores_oob = errores_oob.rename(index={"OOB": "Global"})
errores_oob = errores_oob.reindex(errores_test.index)

# Conserva la tabla numérica para el gráfico siguiente; añade la diferencia en puntos.
tabla_errores_rf = pd.DataFrame({"Test": errores_test, "OOB": errores_oob})
tabla_errores_rf["Test − OOB (pp)"] = 100 * (
    tabla_errores_rf["Test"] - tabla_errores_rf["OOB"]
)

# Presenta los errores como porcentajes, sin alterar los valores numéricos originales.
tabla_errores_rf_presentacion = pd.DataFrame(
    {
        "Error en test": tabla_errores_rf["Test"].map(
            lambda valor: f"{100 * valor:.1f} %"
        ),
        "Error OOB": tabla_errores_rf["OOB"].map(
            lambda valor: f"{100 * valor:.1f} %"
        ),
        "Diferencia (pp)": tabla_errores_rf["Test − OOB (pp)"].map(
            lambda valor: f"{valor:+.1f}"
        ),
    },
    index=tabla_errores_rf.index,
)
display(tabla_errores_rf_presentacion)

In [ ]:
# Compara los errores por categoría en barras horizontales para facilitar la lectura.
orden_grafico = tabla_errores_rf["Test"].sort_values().index
valores_test = tabla_errores_rf.loc[orden_grafico, "Test"].to_numpy()
valores_oob = tabla_errores_rf.loc[orden_grafico, "OOB"].to_numpy()
posiciones = np.arange(len(orden_grafico))
alto_barra = 0.36

fig, ax = plt.subplots(figsize=(9, 5.5))
_ = ax.barh(posiciones - alto_barra / 2, valores_test, alto_barra, label="Test", color="#2878B5")
_ = ax.barh(posiciones + alto_barra / 2, valores_oob, alto_barra, label="OOB", color="#F28E2B")
_ = ax.set(
    yticks=posiciones,
    yticklabels=orden_grafico,
    xlim=(0, 1),
    xlabel="Proporción de error",
    title="Error de clasificación: test frente a OOB",
)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda valor, _: f"{valor:.0%}"))
_ = ax.set_axisbelow(True)
_ = ax.grid(axis="x", alpha=0.25)
_ = ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
plt.show()

#### Matrices de confusión normalizadas

Para comparar **qué clases se confunden entre sí**, ponemos las dos matrices con la misma orientación: **clases reales en filas** y **predicciones en columnas**. La matriz de test se transpone para igualar la orientación de la matriz OOB; después normalizamos cada fila, de modo que cada casilla representa la proporción de predicciones para esa clase real.

Los mapas de calor muestran los porcentajes con la misma escala. La **diagonal** reúne los aciertos de cada clase; las casillas fuera de ella indican con qué otras clases se confunde. Al normalizar por filas comparamos proporciones, no cantidades de productos.

In [ ]:
confusion_oob = oob_rf.confusion  # Recupera los recuentos OOB: clases reales en filas y predicciones en columnas.
test_por_clase = t_rf.T.div(t_rf.T.sum(axis=1), axis=0)  # Transpone test y normaliza cada fila por su clase real.
oob_por_clase = confusion_oob.loc[:, clases].div(
    confusion_oob.loc[:, clases].sum(axis=1), axis=0
)  # Normaliza los recuentos OOB por clase real.

# Compara las distribuciones de predicción con mapas de calor en una escala común.
fig, ejes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
for ax, matriz, titulo in zip(
    ejes,
    [test_por_clase, oob_por_clase],
    ["Test", "OOB"],
):
    imagen = ax.imshow(matriz.to_numpy(), cmap="Blues", vmin=0, vmax=1)
    _ = ax.set(
        xticks=np.arange(len(clases)),
        yticks=np.arange(len(clases)),
        xticklabels=clases,
        yticklabels=clases,
        xlabel="Predicción",
        ylabel="Clase real",
        title=f"Matriz normalizada · {titulo}",
    )
    ax.tick_params(axis="x", labelrotation=45)
    for fila in range(len(clases)):
        for columna in range(len(clases)):
            valor = matriz.iloc[fila, columna]
            _ = ax.text(
                columna,
                fila,
                f"{valor:.0%}",
                ha="center",
                va="center",
                fontsize=7,
                color="white" if valor >= 0.5 else "#222222",
            )

_ = fig.colorbar(
    imagen,
    ax=ejes,
    shrink=0.85,
    format=plt.FuncFormatter(lambda valor, _: f"{valor:.0%}"),
    label="Proporción dentro de la clase real",
)
plt.show()

#### Interpretación de las matrices de confusión

En cada mapa, lee una **fila** para seguir los productos de una clase real y una **columna** para identificar la clase predicha. La casilla diagonal representa el acierto dentro de esa clase; las demás muestran cómo se distribuyen sus errores. Los porcentajes se muestran redondeados a enteros, por lo que un **0 % no siempre implica ausencia total de casos** y una fila puede no sumar exactamente 100 % al visualizarse.

En esta ejecución se observan estos patrones:

- **Clases con alto acierto:** la diagonal de `Class_5` muestra **94 % en test** y **96 % en OOB**; la de `Class_6`, **95 % y 94 %**. También destacan `Class_8` con **93 % y 91 %** y `Class_2` con **89 % y 87 %**.
- **`Class_3` frente a `Class_2`:** aproximadamente el **51 %** de los productos reales de `Class_3` se predice como `Class_2`, tanto en test como en OOB. La diagonal de `Class_3` es **46 % en test** y **45 % en OOB**. Esto no significa que ocurra lo mismo en sentido contrario: de los productos reales de `Class_2`, se predice `Class_3` para el **9 % en test** y el **11 % en OOB**.
- **`Class_4` frente a `Class_2`:** el **44 % en test** y el **43 % en OOB** de los productos reales de `Class_4` se predicen como `Class_2`. Su acierto en la diagonal es **37 % y 34 %**, respectivamente.
- **`Class_1`:** además de los aciertos de su diagonal, aparecen confusiones con `Class_8` (**22 % en test**, **20 % en OOB**) y con `Class_9` (**18 % y 21 %**).

Ambas evaluaciones muestran patrones parecidos en este ejemplo, aunque no idénticos: test utiliza productos reservados y OOB utiliza predicciones de entrenamiento realizadas por los árboles que dejaron fuera cada producto. Estas cifras describen esta muestra y este bosque; no explican por sí solas la causa de las confusiones ni garantizan que se repitan en datos futuros.

### Micropráctica 7.3 — Comparar aciertos OOB y de test (opcional)

Compara, para `Class_1`, la proporción de ejemplos reales clasificados correctamente en test y en OOB. Ambas tablas normalizadas de la celda anterior tienen clases reales en filas y predicciones en columnas. Completa el hueco con la clase indicada para acceder a la diagonal.

**Para pensar:** ¿son parecidos los dos aciertos? ¿Por qué una coincidencia en esta muestra no garantiza el mismo resultado en datos futuros? Esta es una comparación descriptiva, no una selección de modelos con test.

In [ ]:
# Sustituye ... por el nombre de la clase indicada, entre comillas.
clase_mp73 = ...

if clase_mp73 is ...:
    print("Micropráctica 7.3 pendiente: completa la clase.")
else:
    acierto_test_mp73 = test_por_clase.loc[clase_mp73, clase_mp73]
    acierto_oob_mp73 = oob_por_clase.loc[clase_mp73, clase_mp73]
    display(pd.Series({"Acierto en test": acierto_test_mp73,
                       "Acierto OOB": acierto_oob_mp73}, name=clase_mp73))

### ¿Necesitamos más árboles?

El gráfico separa el error OOB **global** y el de cada clase en paneles con la misma escala. El eje horizontal indica cuántos árboles se han acumulado; el vertical muestra el porcentaje de productos mal clasificados entre los que ya tienen votos OOB. El porcentaje del título es el error con todos los árboles del bosque.

Al principio, la cobertura OOB es menor y va creciendo: no todas las filas tienen todavía una predicción OOB. Si las curvas cambian poco al añadir árboles, pueden estar entrando en una zona estable para esta muestra; una curva global estable no implica que todas las clases se comporten igual. Observar solo 50 árboles tampoco demuestra qué ocurriría con un bosque mayor.

In [ ]:
# Un panel por serie evita superponer las nueve clases con el error global.
series_oob = ["OOB"] + list(clases)
fig, ejes = plt.subplots(5, 2, figsize=(11, 12), sharex=True, sharey=True, constrained_layout=True)

for ax, serie in zip(ejes.flat, series_oob):
    errores = oob_rf.curva[serie]
    color = "#2878B5" if serie == "OOB" else "#F28E2B"
    nombre = "Global" if serie == "OOB" else serie
    _ = ax.plot(oob_rf.curva.index, errores, color=color, linewidth=2)
    _ = ax.scatter(oob_rf.curva.index[-1], errores.iloc[-1], color=color, s=25, zorder=3)
    _ = ax.set_title(f"{nombre} · final: {errores.iloc[-1]:.1%}", loc="left", fontsize=11)
    _ = ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda valor, _: f"{valor:.0%}"))
    ax.grid(alpha=0.2)

_ = fig.suptitle("Evolución del error OOB al añadir árboles", fontsize=15)
_ = fig.supxlabel("Número de árboles")
_ = fig.supylabel("Error OOB")
plt.show()

#### Interpretación: evolución del error global

En esta ejecución, el error OOB global baja desde aproximadamente **36 % con un árbol** hasta **21,7 % con 50 árboles**. La reducción es más visible al principio; en el tramo final la curva se aplana y cambia poco. Esto sugiere que el error global se está estabilizando **dentro del intervalo observado**, no que 50 sea el número óptimo de árboles.

La comparación con el primer punto requiere cautela: con pocos árboles, solo se evalúan las filas que ya han recibido algún voto OOB y la cobertura cambia al añadir árboles. Además, el error global resume todas las clases y puede ocultar diferencias importantes entre ellas.

#### Interpretación: diferencias entre clases

Con los **50 árboles** de esta ejecución, los errores OOB dentro de cada clase real son:

- **Menores:** `Class_5` (**4,1 %**), `Class_6` (**6,0 %**) y `Class_8` (**9,5 %**). Sus curvas descienden y presentan pocos cambios al final.
- **Intermedios:** `Class_2` (**13,0 %**) y `Class_9` (**16,7 %**); en ambos casos el error baja al acumular árboles.
- **Mayores:** `Class_7` (**48,1 %**), `Class_3` (**54,8 %**), `Class_1` (**61,6 %**) y `Class_4` (**65,5 %**). La curva de `Class_7` desciende de forma apreciable; las de `Class_1`, `Class_3` y `Class_4` permanecen en niveles relativamente altos, con oscilaciones.

Por ejemplo, el **65,5 % de `Class_4`** significa que, de los productos reales de esa clase con predicción OOB, esa proporción fue clasificada en otra clase. Es un error **dentro de la clase**, no el porcentaje de todos los productos. Los valores corresponden a este bosque y esta muestra; no explican por sí solos por qué unas clases son más difíciles ni prueban que añadir más árboles vaya a resolverlo. Para ver **con qué clases se confunden**, consulta las matrices normalizadas anteriores.

Aunque varias curvas se aplanan al final, algunas todavía cambian: comparar un bosque mayor ayudaría a comprobar si esa tendencia continúa. La evolución observada no garantiza el error futuro.

#### Explorar una curva OOB

Selecciona **Global** o una clase y mueve el deslizador para mostrar la evolución hasta el número de árboles elegido. El punto resalta el último valor mostrado y la tabla resume ese error y la **cobertura global**: la proporción de todas las filas de entrenamiento que ya tiene al menos un voto OOB. La cobertura no es específica de la clase seleccionada.

El control consulta la curva ya calculada: no entrena más árboles ni cambia el bosque. Con pocos árboles, la cobertura es parcial y el error se calcula solo con las filas que ya tienen votos OOB.

In [ ]:
def explorar_oob(clase, arboles):
    # Usa solo los primeros árboles del diagnóstico ya calculado; no ajusta otro bosque.
    curva = oob_rf.curva.iloc[:arboles]
    ultimo = curva.iloc[-1]
    error = ultimo[clase]
    cobertura = ultimo["Cobertura"]
    nombre = "Global" if clase == "OOB" else clase
    color = "#2878B5" if clase == "OOB" else "#F28E2B"

    fig, ax = plt.subplots(figsize=(8, 4))
    _ = ax.plot(curva.index, curva[clase], color=color, linewidth=2)
    _ = ax.scatter(curva.index[-1], error, color=color, s=45, zorder=3)
    _ = ax.set(
        xlim=(1, max(2, len(oob_rf.curva))),
        ylim=(0, 1),
        xlabel="Número de árboles acumulados",
        ylabel="Error OOB",
        title=f"Evolución del error OOB · {nombre}",
    )
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda valor, _: f"{valor:.0%}"))
    ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()

    # La cobertura es global, incluso al seleccionar el error de una clase.
    resumen = pd.DataFrame(
        {
            "Árboles": [arboles],
            "Serie": [nombre],
            "Error OOB": [f"{error:.1%}" if pd.notna(error) else "Sin datos"],
            "Cobertura OOB global": [f"{cobertura:.1%}"],
        }
    )
    display(resumen.set_index("Árboles"))
    return {"error": error, "cobertura": cobertura}

control_oob = widgets.interactive(
    explorar_oob,
    clase=widgets.Dropdown(
        options=[("Global", "OOB")] + [(str(c), c) for c in clases],
        description="Serie",
    ),
    arboles=widgets.IntSlider(
        value=len(oob_rf.curva), min=1, max=len(oob_rf.curva),
        description="Árboles", continuous_update=False,
    ),
)
display(control_oob)

### Importancia de las variables

¿Qué características utiliza este bosque para clasificar? Compararemos dos maneras distintas de resumir su papel. Ninguna demuestra que una variable **cause** la clase del producto.

#### 1. Permutación en observaciones OOB

Para cada árbol y cada predictor, el cálculo compara su acierto en las filas OOB **antes y después de desordenar ese predictor**. Se permuta una variable cada vez, una vez por árbol, sin volver a entrenarlo. La diferencia es:

$$d_{tj}=\text{acierto OOB original del árbol }t-\text{acierto OOB tras permutar la variable }j.$$

Una diferencia positiva indica que ese árbol acertó menos tras la permutación; una negativa, que acertó más en esa permutación. Se calcula también el cambio dentro de cada **clase real**. Después se promedian las diferencias de los árboles, cada uno con el mismo peso. El resultado global considera todas las filas OOB de cada árbol: **no es el cambio de acierto del voto agregado del bosque** ni una media equilibrada de clases.

La tabla `v_imp0` presenta ese descenso medio dividido por su error estándar entre árboles: la columna global se llama `MeanDecreaseAccuracy` y las columnas de clase usan su etiqueta. Es una **puntuación escalada**, no un porcentaje ni puntos porcentuales. Si el error estándar es prácticamente cero, el cálculo conserva el descenso sin escalar. Las pérdidas medias originales se guardan en `v_imp.descensos` y pueden expresarse en puntos porcentuales multiplicando por 100. El error estándar resume variación entre estos árboles; no es una prueba de significación ni toda la incertidumbre del problema.

#### 2. Reducción de impureza Gini

`MeanDecreaseGini` promedia entre árboles la reducción de impureza Gini acumulada y ponderada de las divisiones que usan cada predictor. Se calcula durante el ajuste, **no con las filas OOB**; en esta implementación no está normalizada para sumar 1. Su escala es distinta de la puntuación por permutación, por lo que compararemos posiciones dentro de cada ranking, no sus magnitudes entre sí. Esta medida puede favorecer variables con más posibles puntos de corte.

Una variable correlacionada con otras puede compartir o ceder importancia; al permutarla también se rompe su relación con los demás predictores. Por ello, un valor bajo no prueba que sea inútil: los resultados dependen del bosque, los datos y la permutación. Conservamos las 93 características y las nueve clases para explorarlas.

In [ ]:
v_imp = importancia_oob(rf_mod, X_train, y_train, seed=SEED)  # Calcula importancias con el bosque y las mismas filas de entrenamiento.
v_imp0 = v_imp.tabla  # Conserva todas las variables y medidas numéricas para tablas y controles.
print(f"Importancias calculadas para {len(v_imp0)} predictores y {len(clases)} clases.")
print("Las tablas completas están disponibles en v_imp0, v_imp.descensos y v_imp.error_estandar.")

In [ ]:
# Cada panel ordena las 15 variables con mayor puntuación según su propia medida.
fig, ejes = plt.subplots(1, 2, figsize=(13, 7), constrained_layout=True)
for ax, medida, titulo, etiqueta, color in zip(
    ejes,
    ["MeanDecreaseAccuracy", "MeanDecreaseGini"],
    ["Permutación OOB · global", "Reducción de impureza Gini"],
    ["Descenso medio / error estándar", "Reducción ponderada media"],
    ["#2878B5", "#F28E2B"],
):
    seleccion = v_imp0[medida].nlargest(15).sort_values()
    _ = ax.barh(seleccion.index, seleccion.values, color=color)
    _ = ax.axvline(0, color="#555555", linewidth=0.8)
    _ = ax.set(title=titulo, xlabel=etiqueta)
    ax.set_axisbelow(True)
    ax.grid(axis="x", alpha=0.2)
_ = fig.suptitle("Importancia de predictores · 15 primeros por medida", fontsize=14)
plt.show()

#### Interpretar los rankings globales

En esta ejecución, **`feat_34` encabeza ambos rankings**. En la importancia escalada por permutación le siguen `feat_48` y `feat_62`; en Gini le siguen `feat_60` y `feat_11`. Los paneles ordenan los predictores por separado y usan escalas distintas: que una barra sea más larga en un panel que en el otro no permite comparar directamente sus valores.

La tabla siguiente conserva ese orden por puntuación escalada y añade el descenso medio **sin escalar**, expresado en puntos porcentuales, y la reducción Gini. Por ejemplo, al permutar `feat_34`, la caída media de acierto OOB **por árbol** es aproximadamente **4,35 puntos porcentuales** en esta ejecución; no significa que el bosque agregado pierda exactamente esa cantidad. La puntuación escalada depende también de la variación entre árboles: `feat_11` tiene un descenso medio mayor (**4,74 pp**), aunque su puntuación escalada es menor que la de `feat_34`. Se muestran 15 filas para facilitar la lectura; `orden_global` conserva las 93.

In [ ]:
orden_global = v_imp0.sort_values("MeanDecreaseAccuracy", ascending=False)  # Mantiene el ranking completo.
tabla_global_imp = pd.DataFrame(
    {
        "Permutación global · puntuación escalada": orden_global["MeanDecreaseAccuracy"],
        "Descenso medio de acierto (pp)": 100 * v_imp.descensos.loc[orden_global.index, "MeanDecreaseAccuracy"],
        "Reducción Gini media": orden_global["MeanDecreaseGini"],
    }
)
tabla_global_imp.index.name = "Predictor"
display(tabla_global_imp.head(15).round(2))

#### Importancia para `Class_1`

Para cada árbol se calcula ahora el cambio de acierto **solo entre sus filas OOB cuya clase real es `Class_1`**, tras permutar la variable en todas las filas OOB de ese árbol. La puntuación escalada resume ese cambio entre árboles; el descenso medio en puntos porcentuales muestra el cambio sin escalar.

En esta ejecución destacan **`feat_15`, `feat_17` y `feat_40`**, no las mismas tres variables que lideran el ranking global. Esto ilustra que el papel de una variable puede depender de la clase examinada; no demuestra que esas variables causen la clase ni que las diferencias se mantengan en otras muestras. Se muestran 15 filas y `orden_clase1` conserva el ranking completo.

In [ ]:
orden_clase1 = v_imp0.sort_values("Class_1", ascending=False)  # Mantiene el ranking completo para Class_1.
tabla_clase1_imp = pd.DataFrame(
    {
        "Class_1 · puntuación escalada": orden_clase1["Class_1"],
        "Descenso medio de acierto (pp)": 100 * v_imp.descensos.loc[orden_clase1.index, "Class_1"],
    }
)
tabla_clase1_imp.index.name = "Predictor"
display(tabla_clase1_imp.head(15).round(2))

#### Explorar importancia global o por clase

Elige una medida y cuántos predictores mostrar. El gráfico y la tabla ordenan los mayores valores **de esa medida**; el control no vuelve a ajustar el bosque ni a permutar variables. Las puntuaciones por permutación son escaladas, mientras que Gini usa otra escala. Si aparece un valor negativo de permutación, significa que el acierto medio resultó mayor tras permutar en este cálculo; no implica que la variable sea perjudicial en todos los datos.

In [ ]:
def explorar_importancia(medida, cantidad):
    seleccion = v_imp0[medida].nlargest(cantidad).sort_values()  # Ordena los valores para dibujar el mayor arriba.
    if medida == "MeanDecreaseGini":
        titulo = "Reducción de impureza Gini"
        etiqueta = "Reducción ponderada media"
        color = "#F28E2B"
    else:
        titulo = "Permutación OOB · " + ("global" if medida == "MeanDecreaseAccuracy" else medida)
        etiqueta = "Descenso medio / error estándar"
        color = "#2878B5"

    fig, ax = plt.subplots(figsize=(8, max(4, cantidad * 0.27)), constrained_layout=True)
    _ = ax.barh(seleccion.index, seleccion.values, color=color)
    _ = ax.axvline(0, color="#555555", linewidth=0.8)
    _ = ax.set(title=titulo, xlabel=etiqueta, ylabel="Predictor")
    ax.set_axisbelow(True)
    ax.grid(axis="x", alpha=0.2)
    plt.show()

    tabla = seleccion.sort_values(ascending=False).rename("Valor").to_frame()
    tabla.index.name = "Predictor"
    display(tabla.round(2))
    return seleccion

control_importancia = widgets.interactive(
    explorar_importancia,
    medida=widgets.Dropdown(
        options=[("Permutación · global", "MeanDecreaseAccuracy"),
                 ("Reducción Gini", "MeanDecreaseGini")]
                + [(f"Permutación · {c}", c) for c in clases],
        description="Medida",
        layout=widgets.Layout(width="350px"),
    ),
    cantidad=widgets.IntSlider(value=15, min=5, max=len(predictores), step=1,
                               description="Variables", continuous_update=False),
)
display(control_importancia)

### Ajustar max_features

`max_features` determina cuántas características candidatas se consideran en cada división. En vez de detenernos cuando una mejora es pequeña, compararemos una **cuadrícula de 11 valores**: 2, 5, 9, 12, 15, 18, 24, 32, 48, 64 y 93. Incluimos 9 como referencia de $\lfloor\sqrt{93}\rfloor$ y 93 para considerar todos los predictores.

Para cada valor ajustamos un bosque de **50 árboles** con la misma semilla y únicamente con `X_train` e `y_train`. Comparamos el error OOB global y su cobertura; no se usa `X_test` en la búsqueda. La tabla y el gráfico destacan el menor error observado **entre estos candidatos**. Las pequeñas diferencias pueden variar con otras semillas o muestras: el mínimo no demuestra un óptimo universal.

La búsqueda emplea OOB para seleccionar un candidato; para una evaluación final independiente, reserva test y evita consultarlo repetidamente para decidir. Las comparaciones previas con test en este notebook son descriptivas.

In [ ]:
# Compara todos los valores de la cuadrícula, sin detenerse antes de tiempo.
valores_max_features = [2, 5, 9, 12, 15, 18, 24, 32, 48, 64, 93]
resultados_max_features = []
for valor_max_features in valores_max_features:
    bosque_candidato = RandomForestClassifier(
        n_estimators=50,
        max_features=valor_max_features,
        bootstrap=True,
        min_samples_leaf=1,
        random_state=SEED,
        n_jobs=1,
    ).fit(X_train, y_train)
    diagnostico_candidato = diagnostico_oob(bosque_candidato, X_train, y_train)
    resultado_final = diagnostico_candidato.curva.iloc[-1]
    resultados_max_features.append({
        "max_features": valor_max_features,
        "Error OOB": resultado_final["OOB"],
        "Cobertura OOB": resultado_final["Cobertura"],
    })

max_features_par = pd.DataFrame(resultados_max_features).set_index("max_features")
minimo_oob = max_features_par["Error OOB"].min()
mejores_max_features = max_features_par.index[max_features_par["Error OOB"] == minimo_oob].tolist()
print(f"Menor error OOB observado: {minimo_oob:.2%} · max_features = {mejores_max_features}")
tabla_max_features = max_features_par.copy()
for columna in ["Error OOB", "Cobertura OOB"]:
    tabla_max_features[columna] = tabla_max_features[columna].map(lambda valor: f"{valor:.2%}")
display(tabla_max_features)

fig, ax = plt.subplots(figsize=(9, 4.5))
_ = ax.plot(max_features_par.index, max_features_par["Error OOB"], "o-", color="#2878B5", linewidth=2)
_ = ax.scatter(mejores_max_features, max_features_par.loc[mejores_max_features, "Error OOB"],
               color="#F28E2B", s=80, zorder=3, label="Menor error observado")
_ = ax.set(xticks=valores_max_features, xlabel="max_features · predictores candidatos",
           ylabel="Error OOB", title="Error OOB para cada valor de max_features")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda valor, _: f"{valor:.1%}"))
ax.grid(alpha=0.25)
_ = ax.legend(frameon=False)
fig.tight_layout()
plt.show()

#### Interpretación de la cuadrícula

En esta ejecución, el menor error OOB observado es **21,31 % con `max_features = 32`**. El error baja desde **24,20 % con 2** hasta alrededor del **21,3–21,5 %** para varios valores intermedios; con **93** sube a **22,67 %**. El eje vertical está ampliado y **no empieza en cero**, para que se aprecien las diferencias pequeñas.

Por ejemplo, con **18** el error es **21,34 %**, solo **0,03 puntos porcentuales** más que con 32. Esta diferencia descriptiva no demuestra que 32 vaya a ser mejor en otras muestras o con otra semilla. Los resultados anteriores con todos los productos no son directamente comparables: esta cuadrícula utiliza solo entrenamiento.

#### Ajuste ilustrativo con max_features = 32

A modo de comparación, ajustamos otro bosque con 32 predictores candidatos por división. La matriz y el acierto siguientes usan test para una comparación descriptiva; no deben emplearse para escoger entre configuraciones.

In [ ]:
rf_mod1 = RandomForestClassifier(  # Crea el segundo bosque para comparar con max_features = 9.
    n_estimators=50, max_features=32, bootstrap=True,  # Usa 50 árboles, 32 candidatas por división y bootstrap.
    min_samples_leaf=1, random_state=SEED, n_jobs=1,  # Conserva el tamaño de hoja, semilla y uso de un solo proceso.
).fit(X_train, y_train)  # Ajusta el modelo con la misma muestra de entrenamiento.
pred_rf1 = rf_mod1.predict(X_test)  # Predice las mismas filas de test.
evaluacion_rf1 = evaluar_clasificador(y_test, pred_rf1, clases)  # Resume acierto y matriz.
evaluacion_rf1.confusion  # Muestra la matriz para comparar clases.
print(f"Proporción global de predicciones correctas {round(evaluacion_rf1.acierto * 100, 2)} %") # Muestra el acierto global del nuevo bosque.

#### Inspeccionar la predicción de un producto

El control permite escoger cualquiera de los valores de `max_features` de la cuadrícula anterior y una fila de test. Para ese producto veremos su clase real, la predicción y las probabilidades estimadas para todas las clases.

Los bosques ya ajustados con 9 y 32 se reutilizan. Para los demás valores, la primera selección ajusta un bosque de 50 árboles con la misma configuración de la cuadrícula y únicamente con entrenamiento; puede tardar unos segundos. Una vez ajustado, se conserva en memoria para consultas posteriores.

Las probabilidades son el promedio de las probabilidades de clase de los árboles; no deben interpretarse automáticamente como certeza ni como probabilidades calibradas. Este control usa productos de test para explorar ejemplos, no para escoger el mejor valor de `max_features`.

In [ ]:
# Reutiliza los dos bosques anteriores; los demás se ajustan solo al solicitarlos.
bosques_producto_rf = {rf_mod.max_features: rf_mod, rf_mod1.max_features: rf_mod1}

def explorar_producto_rf(ajuste, producto):
    if ajuste not in bosques_producto_rf:
        print(f"Ajustando bosque con max_features = {ajuste}; puede tardar unos segundos...")
        bosques_producto_rf[ajuste] = RandomForestClassifier(
            n_estimators=50, max_features=ajuste, bootstrap=True,
            min_samples_leaf=1, random_state=SEED, n_jobs=1,
        ).fit(X_train, y_train)
    modelo = bosques_producto_rf[ajuste]
    entrada = X_test.iloc[[producto]]  # Conserva la fila como tabla con nombres de columnas.
    probabilidades = pd.Series(modelo.predict_proba(entrada)[0], index=modelo.classes_, name="Probabilidad")
    prediccion = modelo.predict(entrada)[0]
    display(pd.Series({"Índice": entrada.index[0], "Clase real": y_test.iloc[producto],
                       "Predicción": prediccion, "max_features": ajuste}))
    display(probabilidades.to_frame())
    barras_clases(probabilidades, f"Probabilidad estimada · max_features = {ajuste}", ylabel="Probabilidad")
    return {"modelo": modelo, "probabilidades": probabilidades, "prediccion": prediccion}

control_producto_rf = widgets.interactive(
    explorar_producto_rf,
    ajuste=widgets.Dropdown(
        options=[(f"max_features = {valor}", valor) for valor in valores_max_features],
        value=rf_mod.max_features, description="Bosque", layout=widgets.Layout(width="280px"),
    ),
    producto=widgets.IntSlider(value=0, min=0, max=len(X_test)-1, description="Producto", continuous_update=False),
)
display(control_producto_rf)

## Ejercicio 7-2 — Ampliar el random forest (sin resolver)

Ajusta un bosque de **500 árboles** con todos los productos y estima su error mediante OOB. El error OOB se calcula usando, para cada fila, únicamente los árboles que dejaron esa fila fuera de su muestra bootstrap.

Usa el error por árbol para valorar si hacen falta más árboles: ¿la curva global se estabiliza?, ¿ocurre lo mismo con cada clase? El bosque adicional debe ser una variable distinta, para no sobrescribir `rf_mod` ni desactivar los controles anteriores. La solución queda a tu cargo.

In [ ]:
# Pistas para plantear el ejercicio 7-2 (sin una solución completa):
# Los ejemplos están comentados: cópialos, completa los ... y adáptalos.
# Los nombres con sufijo _ej72 evitan sobrescribir los modelos anteriores.
#
# 1) Prepara los datos:
#    - El enunciado pide TODOS los productos, no solo entrenamiento.
#    - Usa las columnas predictoras para X y la columna objetivo para y.
#
#    X_ej72 = d[...]
#    y_ej72 = d[...]
#    print(X_ej72.shape, y_ej72.shape)
#    print(X_ej72.columns)  # Comprueba que no aparecen id ni target.
#
#    Antes de ajustar, anota qué esperas que ocurra con el error OOB y si
#    crees que todas las clases se estabilizarán al mismo ritmo.
#
# 2) Construye y ajusta un bosque NUEVO:
#    - n_estimators fija el número de árboles; bootstrap debe estar activado.
#    - Elige max_features y justifica tu elección; SEED permite repetir el ajuste.
#
#    bosque_ej72 = RandomForestClassifier(
#        n_estimators=...,
#        max_features=...,
#        bootstrap=True,
#        min_samples_leaf=1,
#        random_state=SEED,
#        n_jobs=1,
#    )
#    bosque_ej72.fit(..., ...)
#    print(len(bosque_ej72.estimators_))  # Comprueba el número de árboles.
#
#    No sobrescribas rf_mod ni rf_mod1. Ajustar 500 árboles puede tardar.
#
# 3) Calcula el diagnóstico OOB y consulta la curva:
#    - Pasa el modelo ajustado y exactamente las mismas filas y el mismo orden
#      usados en .fit(): la función reconstruye las muestras fuera de bolsa.
#
#    diagnostico_ej72 = diagnostico_oob(..., ..., ...)
#    curva_ej72 = diagnostico_ej72.curva
#    display(curva_ej72.head())
#    display(curva_ej72.tail())
#    print(curva_ej72.shape)
#
#    Cada fila corresponde a los primeros k árboles acumulados. La columna
#    "OOB" contiene el error global; cada clase tiene su propia columna y
#    "Cobertura" indica la proporción de filas que ya cuenta con votos OOB.
#
# 4) Crea una tabla breve y dibuja la evolución:
#    - Elige los números de árboles que quieras comparar, incluyendo el final.
#    - Ejemplo de acceso por número de árbol y selección de columnas:
#
#    puntos_ej72 = [...]  # Completa una lista de números de árboles válidos.
#    columnas_ej72 = ["OOB", "Cobertura", ...]  # Añade clases que quieras examinar.
#    display(curva_ej72.loc[puntos_ej72, columnas_ej72])
#
#    - Para el gráfico global usa el índice de la curva y su columna "OOB":
#
#    fig, ax = plt.subplots()
#    ax.plot(curva_ej72.index, curva_ej72[...])
#    ax.set(xlabel="Número de árboles", ylabel="Error OOB", title="...")
#    ax.grid(alpha=0.25)
#    plt.show()
#
#    - Para una clase, accede por su nombre, por ejemplo curva_ej72["Class_1"].
#      Puedes recorrer las clases o usar paneles separados; compara todas las
#      clases, no solo el ejemplo. Los errores son proporciones entre 0 y 1.
#
# 5) Interpreta sin dar por supuesto el resultado:
#    - Compara cambios y oscilaciones del tramo final, no solo dos puntos.
#    - ¿Qué clases mantienen errores mayores? ¿Se estabilizan igual que el global?
#    - Con pocos árboles la cobertura puede ser parcial; el error solo usa filas
#      con votos OOB. Una curva aparentemente estable no garantiza el resultado
#      con más árboles ni en otra muestra.
#
# Al entrenar con todos los productos, la partición test anterior participa en
# este bosque: este ejercicio es exploratorio y no tiene un test independiente.

## 7.3. Máquinas de vectores soporte: SVM

Una **SVM** aprende una frontera que separa clases intentando mantener un margen amplio. Los productos próximos a la frontera que influyen en el ajuste se llaman **vectores soporte**. Con datos que no se separan perfectamente, se permiten violaciones del margen: `C` controla cuánto se penalizan. Un `C` mayor suele exigir un ajuste más estricto a entrenamiento; uno menor admite más violaciones. Ninguno garantiza por sí solo el mejor resultado en datos nuevos.

Un **kernel** define una medida de similitud y permite construir distintas fronteras: lineal, polinómica, radial o sigmoide. En el radial, `gamma` controla cómo disminuye la influencia con la distancia: valores grandes producen influencia más localizada. Compararemos configuraciones con validación cruzada, no seleccionando por test.

### Submuestra y respuesta binaria

Para que las búsquedas sean manejables, seleccionamos **4 000 productos** sin reemplazo. Transformamos la respuesta: **True** significa `Class_5` y **False**, cualquiera de las otras ocho clases. Este problema binario no es directamente comparable con la clasificación de nueve clases de los bloques anteriores.

Asignamos cada fila a entrenamiento o test con probabilidad 0,5: los tamaños no son necesariamente idénticos y la partición **no está estratificada**. El desbalance puede hacer que el acierto global oculte errores en `Class_5`; examinaremos también los aciertos y errores de cada grupo. Usamos variables propias del bloque para conservar los controles anteriores.

In [ ]:
rng_svm = np.random.default_rng(SEED)  # Generador independiente y reproducible para esta submuestra.
seleccion_svm = rng_svm.choice(len(d), size=4000, replace=False)
d1 = d.iloc[seleccion_svm].copy()
d1["target2"] = d1["target"].eq("Class_5")  # True: Class_5; False: resto.
d1 = d1.drop(columns="target")  # La etiqueta original no se usa como predictor.
train_sel_svm = rng_svm.choice([False, True], size=len(d1), p=[0.5, 0.5])
train_svm = d1.loc[train_sel_svm].copy()
test_svm = d1.loc[~train_sel_svm].copy()
X_train_svm, y_train_svm = train_svm[predictores], train_svm["target2"]
X_test_svm, y_test_svm = test_svm[predictores], test_svm["target2"]

# Los recuentos muestran el desbalance y la variación de tamaños de la partición.
conteos_svm = pd.DataFrame({"Entrenamiento": y_train_svm.value_counts(),
                            "Test": y_test_svm.value_counts()}).reindex([False, True], fill_value=0)
conteos_svm.index = pd.Index(["Resto (False)", "Class_5 (True)"], name="Clase real")
display(conteos_svm)
print(f"Acierto en test prediciendo siempre 'resto': {(~y_test_svm).mean():.2%}")

### Ajustar un modelo radial con C = 1

Las SVM dependen de la escala de los predictores. El `Pipeline` combina `StandardScaler` y el clasificador: el escalador aprende medias y desviaciones del conjunto que recibe al ajustarse y transforma después los datos con esos parámetros. Dentro de cada pliegue de validación cruzada se vuelve a ajustar solo con la parte de entrenamiento de ese pliegue; así evitamos usar datos de validación para calcular las escalas.

El kernel radial usa la similitud $K(x,z)=\exp(-\gamma\lVert x-z\rVert^2)$ sobre las características escaladas. En este ejemplo fijamos `C=1` y `gamma=1/93`; son valores de partida, no necesariamente óptimos. El resumen también muestra grado 3 y término independiente 0, aunque **no intervienen en el kernel radial**. Los vectores soporte son observaciones de entrenamiento usadas por el modelo; su número no mide directamente el acierto.

[Documentación de SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html).

In [ ]:
from s7_svm import construir_svm, resumen_busqueda, resumen_svm  # Importa creación del pipeline y resúmenes de SVM.

In [ ]:
mod_svm = construir_svm(len(predictores), kernel="rbf", C=1).fit(X_train_svm, y_train_svm)  # Construye y ajusta SVM radial con C=1.
resumen_svm(mod_svm)  # Resume kernel, parámetros y número de vectores soporte.

### Capacidad predictiva: no solo acierto global

Evaluamos el modelo radial inicial sobre test. La matriz presenta **predicciones en filas y clases reales en columnas**. Para `Class_5` (`True`), los **verdaderos positivos** son aciertos en la diagonal; los **falsos negativos** son `True` reales predichos como `False`. Los **falsos positivos** son `False` reales predichos como `True`.

La **sensibilidad** es la proporción de `Class_5` reales detectados; la **precisión**, la proporción de predicciones `True` que eran realmente `Class_5`. La **especificidad** mide el acierto dentro del resto. El acierto global puede ser alto simplemente porque el grupo `False` es mayoritario.

Las evaluaciones de varios modelos sobre este mismo test son **descriptivas**. Si elegimos el modelo final según estas cifras, test deja de ser una evaluación final independiente; selecciona con entrenamiento y reserva otro conjunto no consultado para una evaluación final.

In [ ]:
def presentar_evaluacion_svm(resultado, nombre):
    """Presenta métricas binarias y recuentos con predicciones en filas."""
    matriz = resultado.confusion.reindex(index=[False, True], columns=[False, True], fill_value=0)
    vn, fn = matriz.loc[False, False], matriz.loc[False, True]
    fp, vp = matriz.loc[True, False], matriz.loc[True, True]
    sensibilidad = vp / (vp + fn) if vp + fn else np.nan
    especificidad = vn / (vn + fp) if vn + fp else np.nan
    precision = vp / (vp + fp) if vp + fp else np.nan
    tabla = pd.DataFrame({"Resultado": [f"{v:.2%}" if pd.notna(v) else "No definido" for v in
        [resultado.acierto, sensibilidad, especificidad, precision]]},
        index=["Acierto global", "Sensibilidad · Class_5", "Especificidad · resto", "Precisión · Class_5"])
    display(tabla)

    fig, ax = plt.subplots(figsize=(5.5, 4))
    imagen = ax.imshow(matriz.to_numpy(), cmap="Blues")
    _ = ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Resto", "Class_5"],
               yticklabels=["Resto", "Class_5"], xlabel="Clase real", ylabel="Predicción", title=nombre)
    maximo = matriz.to_numpy().max()
    for i in range(2):
        for j in range(2):
            valor = matriz.iloc[i, j]
            _ = ax.text(j, i, f"{valor:,}", ha="center", va="center", fontsize=12,
                        color="white" if maximo and valor > maximo / 2 else "#222222")
    _ = fig.colorbar(imagen, ax=ax, label="Productos")
    fig.tight_layout()
    plt.show()
    return tabla

pr_svm = mod_svm.predict(X_test_svm)
evaluacion_svm = evaluar_clasificador(y_test_svm, pr_svm, [False, True])
_ = presentar_evaluacion_svm(evaluacion_svm, "SVM radial inicial · test")

#### Interpretación del modelo radial inicial

En esta partición, **1 927 de 2 013 productos de test** son del grupo resto: predecir siempre resto ya acertaría aproximadamente **95,73 %**, pero no detectaría ningún `Class_5`. Por eso la matriz y la sensibilidad aportan información que el acierto global no muestra.

Con el modelo radial inicial se observan **74 verdaderos positivos, 12 falsos negativos y 3 falsos positivos**. El acierto global es **99,25 %**, mientras que la sensibilidad de `Class_5` es **74 / 86 = 86,05 %**: 12 de los 86 productos reales de esa clase no se detectaron. Son resultados de esta partición, no una garantía de rendimiento futuro.

### Ajustar C con un kernel lineal

Probamos siete valores de `C` con **validación cruzada de diez pliegues** sobre entrenamiento. En cada vuelta, el pipeline ajusta el escalador y la SVM con nueve pliegues y mide el acierto en el décimo; se calcula la media de los diez errores. `GridSearchCV` elige el mayor acierto medio y reajusta ese candidato con todo el entrenamiento (`refit=True`).

La búsqueda usa `accuracy`, aunque la clase positiva es minoritaria: según el objetivo también puede interesar seleccionar con sensibilidad, F1 u otra métrica. Aquí usamos `KFold` mezclado y reproducible, **sin estratificación**; los pliegues pueden contener distintas proporciones de clases. Una alternativa es `StratifiedKFold`.

La tabla y el gráfico muestran **error CV medio** y **desviación estándar entre pliegues (DE)**. Una barra de media ± DE describe variabilidad entre pliegues, no es un intervalo de confianza ni un error estándar. La búsqueda de `C` no usa test.

In [ ]:
costes = [0.01, 0.2, 0.1, 1, 5, 10, 100]  # Enumera los siete valores de C que vamos a comparar.
cv_svm = KFold(n_splits=10, shuffle=True, random_state=SEED)  # Crea diez particiones aleatorias reproducibles.
mod_tune = GridSearchCV(  # Configura una búsqueda exhaustiva sobre los valores indicados.
    construir_svm(len(predictores), "linear"),  # Pasa el pipeline: escala y clasificador lineal.
    {"svm__C": costes},  # Cambia el parámetro C del paso llamado svm.
    cv=cv_svm, scoring="accuracy", refit=True, n_jobs=1, error_score="raise",  # Evalúa acierto y vuelve a ajustar el mejor.
)
with threadpool_limits(limits=1):  # Limita hilos durante el ajuste de los diez pliegues.
    mod_tune.fit(X_train_svm, y_train_svm)  # Busca C usando solo las filas de entrenamiento.

resumen_busqueda(mod_tune)  # Presenta error y variabilidad CV para cada candidato.
mod_tune.best_params_  # Muestra el valor de C con mayor acierto medio.

In [ ]:
def presentar_cv_svm(ajuste, parametro, titulo):
    """Compara error medio CV y desviación entre pliegues sin volver a ajustar."""
    resultados = ajuste.cv_results_
    tabla = pd.DataFrame({
        "Candidato": [p[parametro] for p in resultados["params"]],
        "Error CV medio": 1 - resultados["mean_test_score"],
        "DE entre pliegues": resultados["std_test_score"] * np.sqrt(ajuste.n_splits_ / (ajuste.n_splits_ - 1)),
    })
    if parametro == "svm__C":
        tabla = tabla.sort_values("Candidato")
    tabla = tabla.set_index("Candidato")
    presentacion = tabla.copy()
    for columna in presentacion:
        presentacion[columna] = presentacion[columna].map(lambda v: f"{100 * v:.2f} %")
    display(presentacion)
    print(f"Seleccionado por mayor acierto medio CV: {ajuste.best_params_}")

    fig, ax = plt.subplots(figsize=(8, max(3.2, 0.55 * len(tabla))), constrained_layout=True)
    posiciones_cv = np.arange(len(tabla))
    _ = ax.barh(posiciones_cv, 100 * tabla["Error CV medio"], xerr=100 * tabla["DE entre pliegues"],
                capsize=3, color="#2878B5", alpha=0.85, label="Error medio ± 1 DE")
    _ = ax.set(yticks=posiciones_cv, yticklabels=[str(v) for v in tabla.index],
               xlabel="Error CV (%)", ylabel="C" if parametro == "svm__C" else "Kernel", title=titulo)
    ax.invert_yaxis()
    ax.set_axisbelow(True)
    ax.grid(axis="x", alpha=0.2)
    _ = ax.legend(frameon=False)
    plt.show()
    return tabla

_ = presentar_cv_svm(mod_tune, "svm__C", "SVM lineal · validación cruzada")

#### Interpretación de la búsqueda lineal

En esta ejecución, la búsqueda selecciona **`C = 0,1`** por su acierto medio de validación cruzada. El gráfico permite comparar todos los valores y la variación entre pliegues; diferencias pequeñas entre medias no prueban una ventaja estable. El valor elegido depende de esta cuadrícula, de la partición de entrenamiento y de la métrica utilizada.

### Micropráctica 7.4 — Leer la búsqueda de SVM (opcional)

Recupera el valor de `C` seleccionado para el kernel lineal y compáralo con la tabla de validación cruzada. Sustituye el hueco por la clave del parámetro dentro del pipeline.

**Para pensar:** ¿qué representa la media de validación cruzada y por qué el escalador debe ajustarse dentro de cada pliegue? La búsqueda ya está entrenada: no es necesario volver a ajustarla ni consultar test.

In [ ]:
# Sustituye ... por la clave de C en mod_tune.best_params_, entre comillas.
# Pista: el paso del clasificador se llama svm y sus parámetros usan dos guiones bajos.
clave_c_mp74 = ...

if clave_c_mp74 is ...:
    print("Micropráctica 7.4 pendiente: completa la clave del parámetro.")
else:
    mejor_c_mp74 = mod_tune.best_params_[clave_c_mp74]
    print("C seleccionado:", mejor_c_mp74)
    print("Acierto medio CV:", mod_tune.best_score_)
    display(resumen_busqueda(mod_tune))

In [ ]:
mod_best = mod_tune.best_estimator_  # Recupera el pipeline lineal reajustado con todo el entrenamiento.
resumen_svm(mod_best)  # Muestra sus parámetros y cuántos vectores soporte utiliza.

#### Evaluar el modelo lineal elegido

Una vez seleccionado `C` con entrenamiento, usamos el estimador reajustado para predecir la muestra test. Esta evaluación no participa en la elección del parámetro.

In [ ]:
pr_lineal = mod_best.predict(X_test_svm)
evaluacion_lineal = evaluar_clasificador(y_test_svm, pr_lineal, [False, True])
_ = presentar_evaluacion_svm(evaluacion_lineal, "SVM lineal seleccionada · test")

**Interpretación en este test.** La SVM lineal seleccionada acierta **76 de 86** productos `Class_5` (**88,37 % de sensibilidad**), deja escapar **10** y asigna `Class_5` por error a **3** productos del resto. Su acierto global es **99,35 %**; las cifras describen esta muestra y no sirven para escoger el modelo final con test.

### Kernel polinómico

El kernel polinómico utiliza $K(x,z)=(\gamma\,x^Tz+r)^d$ sobre los predictores escalados, permitiendo fronteras no lineales mediante combinaciones de características. Aquí fijamos **grado 3**, $\gamma=1/93$ y $r=0$; solo variamos `C` con los mismos pliegues y valores candidatos que antes. La comparación cambia el kernel, pero no explora todos sus posibles parámetros.

La búsqueda emplea la misma métrica y valida únicamente con entrenamiento. Observa si el mejor valor está en un extremo de la cuadrícula: en ese caso, no se ha explorado más allá.

In [ ]:
mod_tune1 = GridSearchCV(  # Crea una búsqueda independiente para el kernel polinómico.
    construir_svm(len(predictores), "poly"),  # Construye el pipeline con escalado y kernel poly.
    {"svm__C": costes},  # Reutiliza los mismos valores candidatos de C.
    cv=cv_svm, scoring="accuracy", refit=True, n_jobs=1, error_score="raise",  # Usa los mismos pliegues y vuelve a ajustar el mejor.
)
with threadpool_limits(limits=1):  # Limita hilos durante la búsqueda.
    mod_tune1.fit(X_train_svm, y_train_svm)  # Ajusta y evalúa cada valor de C con entrenamiento.

resumen_busqueda(mod_tune1)  # Muestra los errores CV de cada valor de C.
mod_best1 = mod_tune1.best_estimator_  # Guarda el mejor pipeline polinómico ya reajustado.
resumen_svm(mod_best1)  # Resume los parámetros y vectores soporte del modelo elegido.

In [ ]:
_ = presentar_cv_svm(mod_tune1, "svm__C", "SVM polinómica · validación cruzada")

**Interpretación.** En esta ejecución, el mejor candidato polinómico es **`C = 100`**, el extremo superior de la cuadrícula. La búsqueda no ha probado valores mayores, así que no podemos concluir dónde estaría el mínimo fuera de ese rango. La DE muestra variación entre pliegues y no prueba significación de las diferencias.

#### Evaluar el modelo polinómico elegido

Una vez seleccionado `C` con entrenamiento, usamos el estimador reajustado para predecir la muestra test. Esta evaluación no participa en la elección del parámetro.

In [ ]:
pr_polinomico = mod_best1.predict(X_test_svm)
evaluacion_polinomica = evaluar_clasificador(y_test_svm, pr_polinomico, [False, True])
_ = presentar_evaluacion_svm(evaluacion_polinomica, "SVM polinómica seleccionada · test")

**Interpretación en este test.** La SVM polinómica detecta **73 de 86** productos `Class_5` (**84,88 % de sensibilidad**), deja escapar **13** y produce **12 falsos positivos**. Su acierto global es **98,76 %**. La selección de `C` se realizó con CV; estos resultados de test son descriptivos.

### Kernel radial

El kernel radial utiliza la distancia entre observaciones escaladas y puede construir fronteras no lineales. Comparamos los mismos candidatos de `C` con los mismos diez pliegues, manteniendo **`gamma = 1/93` fijo**. Por tanto, esta búsqueda elige `C` dentro de la cuadrícula, pero no optimiza `gamma`.

In [ ]:
mod_tune2 = GridSearchCV(  # Crea la búsqueda para el kernel radial.
    construir_svm(len(predictores), "rbf"),  # Construye el pipeline con escalado y kernel radial.
    {"svm__C": costes},  # Compara los mismos valores de C que antes.
    cv=cv_svm, scoring="accuracy", refit=True, n_jobs=1, error_score="raise",  # Usa los mismos pliegues y vuelve a ajustar el mejor.
)
with threadpool_limits(limits=1):  # Limita hilos durante el cálculo de las diez particiones.
    mod_tune2.fit(X_train_svm, y_train_svm)  # Ajusta y evalúa cada valor de C con entrenamiento.

resumen_busqueda(mod_tune2)  # Muestra el error CV por candidato.
mod_best2 = mod_tune2.best_estimator_  # Guarda el mejor pipeline radial.
resumen_svm(mod_best2)  # Resume los parámetros del estimador final.

In [ ]:
_ = presentar_cv_svm(mod_tune2, "svm__C", "SVM radial · validación cruzada")

**Interpretación.** En esta ejecución se selecciona **`C = 10`** por el acierto medio CV con el `gamma` fijado. El error medio y la variación entre pliegues describen esta búsqueda; no demuestran que la misma configuración sea óptima para otros datos.

#### Evaluar el modelo radial elegido

Una vez seleccionado `C` con entrenamiento, usamos el estimador reajustado para predecir la muestra test. Esta evaluación no participa en la elección del parámetro.

In [ ]:
pr_radial = mod_best2.predict(X_test_svm)
evaluacion_radial = evaluar_clasificador(y_test_svm, pr_radial, [False, True])
_ = presentar_evaluacion_svm(evaluacion_radial, "SVM radial seleccionada · test")

**Interpretación en este test.** La SVM radial seleccionada detecta **79 de 86** productos `Class_5` (**91,86 % de sensibilidad**), deja escapar **7** y produce **1 falso positivo**. El acierto global es **99,60 %**. Estas cifras corresponden a esta partición; comparar modelos repetidamente en test no proporciona una evaluación final independiente del modelo escogido con esos resultados.

### ¿Qué kernel escoger con C = 1?

Ahora fijamos `C = 1` y comparamos kernels lineal, polinómico, radial y sigmoide con los mismos pliegues. Así estudiamos el efecto de la forma de la frontera manteniendo constante el coste.

La selección de esta búsqueda sigue usando entrenamiento. El resultado de validación cruzada ayuda a comparar candidatos, pero no garantiza que el kernel con mejor media sea el mejor en otras muestras.

In [ ]:
mod_tune3 = GridSearchCV(  # Crea una búsqueda para comparar kernels con C fijado.
    construir_svm(len(predictores), C=1),  # Construye el pipeline con C=1 y kernel por defecto.
    {"svm__kernel": ["linear", "poly", "rbf", "sigmoid"]},  # Enumera las cuatro formas de frontera.
    cv=cv_svm, scoring="accuracy", refit=True, n_jobs=1, error_score="raise",  # Conserva pliegues, medida y ajuste del mejor.
)
with threadpool_limits(limits=1):  # Limita hilos durante la validación.
    mod_tune3.fit(X_train_svm, y_train_svm)  # Compara los kernels únicamente con entrenamiento.

resumen_busqueda(mod_tune3)  # Muestra el error de validación de cada kernel.
mod_tune3.best_params_  # Indica el kernel con mejor acierto medio.
mod_best3 = mod_tune3.best_estimator_  # Guarda el mejor pipeline reajustado con todo el entrenamiento.
resumen_svm(mod_best3)  # Resume el kernel elegido y el número de vectores soporte.

In [ ]:
_ = presentar_cv_svm(mod_tune3, "svm__kernel", "Kernels con C = 1 · validación cruzada")

**Interpretación.** En esta ejecución se selecciona el kernel **radial** con `C = 1`, pero su error CV es muy próximo al del lineal. La selección indica el mejor valor medio observado en estos pliegues, no una diferencia estadísticamente probada ni una garantía para otras muestras. El resto de los parámetros del kernel permanecen fijos.

In [ ]:
pr_kernel = mod_best3.predict(X_test_svm)
evaluacion_kernel = evaluar_clasificador(y_test_svm, pr_kernel, [False, True])
_ = presentar_evaluacion_svm(evaluacion_kernel, "Kernel seleccionado con C = 1 · test")

**Interpretación en este test.** La búsqueda con `C = 1` seleccionó el kernel radial; al reajustarlo con el mismo entrenamiento y los mismos parámetros obtiene aquí las mismas predicciones que el radial inicial: **74 aciertos y 12 fallos** entre los 86 productos `Class_5`. La coincidencia corresponde a esta ejecución y no implica que todos los kernels sean equivalentes.

#### Explorar las búsquedas CV

El menú permite volver a consultar las cuatro búsquedas **ya ajustadas**. Cada barra muestra el error medio en validación cruzada y el segmento negro, **± una desviación estándar entre pliegues**. Los valores de `C` se presentan en orden numérico; para la comparación de kernels, `C` permanece fijo en 1.

La tabla presenta errores y DE como porcentajes. La búsqueda elige el menor error medio entre los candidatos probados, pero la variación entre pliegues no es una prueba de que un candidato sea superior. El control no vuelve a ajustar modelos ni usa test.

In [ ]:
# El diccionario guarda la búsqueda y el parámetro representado en su gráfico.
busquedas_svm = {
    "Lineal: C": (mod_tune, "svm__C"),
    "Polinómico: C": (mod_tune1, "svm__C"),
    "Radial: C": (mod_tune2, "svm__C"),
    "Kernel: C = 1": (mod_tune3, "svm__kernel"),
}

def explorar_cv_svm(busqueda):
    ajuste, parametro = busquedas_svm[busqueda]
    return presentar_cv_svm(ajuste, parametro, f"{busqueda} · validación cruzada")

control_cv_svm = widgets.interactive(
    explorar_cv_svm,
    busqueda=widgets.Dropdown(options=list(busquedas_svm), description="Búsqueda"),
)
display(control_cv_svm)

#### Explorar la función de decisión

Un modelo SVM usa los **93 predictores** y no podemos dibujar ese espacio completo. El control escoge dos características como ejes y fija las otras 91 en sus medianas de entrenamiento. El fondo representa la clase predicha en ese **corte**; los puntos son una muestra reproducible de productos de test coloreados según su clase real.

**Cuidado al leer el gráfico:** cada punto conserva sus valores reales en las otras 91 variables, mientras que el fondo las fija en la mediana. Un punto sobre un fondo de otro color **no implica necesariamente un error**. Además, el gráfico usa límites basados en percentiles, por lo que algunos puntos pueden quedar fuera. La matriz y el acierto del control sí utilizan los 93 predictores reales.

Cambiar las opciones no reajusta los modelos, pero vuelve a mostrar métricas de test. Usa el control para explorar, no para elegir el modelo final con test.

In [ ]:
from s7_graficos import dibujar_decision_svm  # Importa la función de apoyo para dibujar un corte bidimensional.

modelos_svm = {"Radial inicial": mod_svm, "Lineal elegido": mod_best, "Polinómico elegido": mod_best1,  # Reúne los pipelines ya ajustados.
               "Radial elegido": mod_best2, "Kernel elegido con C=1": mod_best3}  # Añade los dos modelos radiales y el kernel comparado.

def explorar_decision_svm(modelo, eje_x, eje_y):  # Define la actualización al escoger modelo y características.
    if eje_x == eje_y:  # Comprueba que no se intente dibujar dos veces el mismo predictor.
        display("Selecciona dos características distintas.")  # Explica el problema en la salida del control.
        return  # Termina la actualización sin dibujar un gráfico inválido.
    ajuste = modelos_svm[modelo]  # Recupera el modelo elegido en el menú.
    prediccion = ajuste.predict(X_test_svm)  # Predice test usando todas las características.
    resultado = evaluar_clasificador(y_test_svm, prediccion, [False, True])  # Calcula matriz y acierto de test.
    display(resumen_svm(ajuste), resultado.confusion, pd.Series({"Proporción de aciertos": resultado.acierto}))  # Muestra el modelo y su evaluación.
    dibujar_decision_svm(ajuste, X_train_svm, X_test_svm, y_test_svm,  # Pasa modelo y datos a la función de dibujo.
                         predictores, eje_x, eje_y, seed=SEED, etiqueta=modelo)  # Indica ejes, semilla y nombre del modelo.

control_decision_svm = widgets.interactive(  # Crea los menús que controlan la visualización y evaluación.
    explorar_decision_svm,  # Conecta los controles con la función definida arriba.
    modelo=widgets.Dropdown(options=list(modelos_svm), value="Lineal elegido", description="Modelo"),  # Elige un modelo ya ajustado.
    eje_x=widgets.Dropdown(options=predictores, value="feat_34", description="Eje X"),  # Elige la característica horizontal.
    eje_y=widgets.Dropdown(options=predictores, value="feat_83", description="Eje Y"),  # Elige la característica vertical.
)
display(control_decision_svm)  # Muestra controles, evaluación y gráfico de decisión.

## Ejercicio 7-3 — SVM multiclase

**Objetivo:** clasificar las nueve clases del producto en lugar de distinguir solo `Class_5` del resto. Usa una submuestra algo mayor, conserva una partición de entrenamiento y test, y crea un `Pipeline` que estandarice antes de ajustar la SVM.

Con `GridSearchCV`, prueba una cuadrícula de hiperparámetros (por ejemplo, kernel, `C` y `gamma`) con validación cruzada **solo en entrenamiento**. El escalado debe permanecer dentro del pipeline para que cada pliegue aprenda sus propias medias y desviaciones. Una vez elegida la configuración, reajústala con todo el entrenamiento y evalúa una sola vez en test.

Incluye la matriz de confusión, el acierto global y el acierto por clase. Compara la dificultad de las nueve clases y explica cómo elegiste la cuadrícula. La solución queda a tu cargo.

In [ ]:
# Pistas para el ejercicio 7-3:
# Los ejemplos están comentados: completa los ... y adapta los nombres.
#
# 1) Prepara una submuestra multiclase:
#    - Parte de d, donde target conserva las nueve clases originales.
#    - Selecciona filas sin reemplazo y conserva una semilla reproducible.
#    - Separa entradas y respuesta sin incluir target entre los predictores.
#
#    rng_ej73 = np.random.default_rng(SEED)
#    indices_ej73 = rng_ej73.choice(len(d), size=..., replace=False)
#    muestra_ej73 = d.iloc[indices_ej73].copy()
#    X_ej73 = muestra_ej73[predictores]
#    y_ej73 = muestra_ej73["target"]
#    print(y_ej73.value_counts().sort_index())
#
# 2) Divide entrenamiento y test, con nombres distintos de los anteriores.
#    Considera una partición estratificada para conservar la representación
#    de las clases y comprueba sus recuentos en ambos conjuntos.
#
#    from sklearn.model_selection import train_test_split, StratifiedKFold
#    X_train_ej73, X_test_ej73, y_train_ej73, y_test_ej73 = train_test_split(
#        X_ej73, y_ej73, test_size=..., random_state=SEED, stratify=y_ej73
#    )
#
# 3) Construye un pipeline con StandardScaler y SVC; el escalador debe quedar
#    DENTRO del pipeline, para ajustarse solo con el entrenamiento de cada pliegue.
#    Consulta cómo nombrar los pasos y usa ese nombre con dos guiones bajos al
#    referirte a parámetros en GridSearchCV. Por ejemplo, si el paso se llama
#    "svm", la clave de C será "svm__C".
#
#    from sklearn.pipeline import Pipeline
#    from sklearn.preprocessing import StandardScaler
#    from sklearn.svm import SVC
#    pipeline_ej73 = Pipeline([("escala", StandardScaler()), ("svm", SVC())])
#    cuadricula_ej73 = {"svm__kernel": [...], "svm__C": [...], "svm__gamma": [...]}
#    cv_ej73 = StratifiedKFold(n_splits=..., shuffle=True, random_state=SEED)
#    busqueda_ej73 = GridSearchCV(pipeline_ej73, cuadricula_ej73,
#                               cv=cv_ej73, scoring="accuracy", refit=True)
#    busqueda_ej73.fit(..., ...)  # Solo entrenamiento.
#
#    Elige una cuadrícula manejable: probar muchas combinaciones y pliegues
#    puede tardar. No todos los parámetros afectan por igual a todos los kernels.
#
# 4) Inspecciona la tabla CV y la configuración elegida sin consultar test:
#    busqueda_ej73.best_params_
#    busqueda_ej73.best_score_
#    pd.DataFrame(busqueda_ej73.cv_results_)[["params", "mean_test_score"]]
#
# 5) Evalúa UNA vez el modelo elegido con la muestra test reservada:
#    pred_ej73 = busqueda_ej73.best_estimator_.predict(...)
#    resultado_ej73 = evaluar_clasificador(..., pred_ej73, clases)
#    display(resultado_ej73.confusion, resultado_ej73.por_clase)
#    print(resultado_ej73.acierto)
#
#    Las filas de la matriz son predicciones y las columnas son clases reales.
#    Compara acierto global y por clase; describe qué etiquetas se confunden.
#    Justifica los parámetros probados y no escojas configuraciones mirando test.